**Tools evaluated:** JACUSA2, Drummer, ELIGOS, Tombo, Yanocomp, Nanocompore, Xpore, EpiNano, nanoDoc, Differr

**Coverage levels:** 5x–1000x (25 levels) and nanoDoc only 15 levels

**References:** 16S rRNA, 23S rRNA

### Figures in this notebook: Benchmarking Tools for Identification of rRNA Modifications

| Figure | Description |
|--------|------------|
| **Fig. 3** | Faceted (3x3) AUROC plots (16S and 23S) |
| **Supp. Fig. S3** | AUROC vs Coverage - two panel |
| **Fig. 4** | AUPRC vs Coverage - two panel |
| **Supp. Fig. S4** | fold over random AUPRC vs Coverage |
| **Fig. 5** | No call rate and Ground-truth recall at different coverages (16S and 23S) |
| **Fig. 6** | Directional offset analysis (16S and 23S)|
| **Fig. 7** | Precision vs Recall at optimal F1 (16S and 23S) |
| **Fig. 8** | Recall vs fp at optimal F1 (16S and 23S)|
| **Fig. 9** | Site-level GT recovery |
| **Supp. Fig. S5** | site-level replicate plots - 16S |
| **Supp. Fig. S6** | site-level replciate plots - 23S |
| **Supp. Fig. S9** | Window analysis |

---

### Requirements

- **Python** 3.10+ with `matplotlib`, `pandas`, `numpy`
- **Data**: `RUN_DIR` (set in the configuration cell) must point to the collated metrics directory containing `metrics_summary_long.csv`, `metrics_long.csv`, and `lag_metrics_summary_long.csv`
- **External modules**: The GT recovery section imports from `downstream_analysis/` (`parse_outputs`, `position_standardization`)
- **Coordinate convention**: rRNA references use 200 nt 5' padding; all positions are converted to native (unpadded) coordinates before evaluation


## Cell 1: Imports and universal style setup

In [ ]:
from collections import defaultdict
from pathlib import Path

import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


# -----------------------------------------------------------------------------
# Figure style settings - for consistent styling across all plots
# -----------------------------------------------------------------------------
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 9,
    'legend.title_fontsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.transparent': False,
    'axes.linewidth': 1.0,
    'lines.linewidth': 1.8,
    'lines.markersize': 5,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# -----------------------------------------------------------------------------
# Tool Configuration - Colors and Markers (Colorblind-Friendly Palette)
# -----------------------------------------------------------------------------
TOOL_ORDER = ['jacusa2', 'drummer', 'eligos', 'tombo', 'yanocomp',
              'nanocompore', 'xpore', 'epinano', 'nanodoc', 'differr']

TOOL_COLORS = {
    'jacusa2':     '#E69F00',  # Orange
    'drummer':     '#56B4E9',  # Sky blue
    'eligos':      '#009E73',  # Bluish green
    'tombo':       '#F0E442',  # Yellow
    'yanocomp':    '#0072B2',  # Blue
    'nanocompore': '#D55E00',  # Vermillion
    'xpore':       '#CC79A7',  # Reddish purple
    'epinano':     '#999999',  # Gray
    'nanodoc':     '#000000',  # Black
    'differr':     '#882255',  # Wine
}

TOOL_MARKERS = {
    'jacusa2': 'o', 'drummer': 's', 'eligos': '^', 'tombo': 'D', 'yanocomp': 'v',
    'nanocompore': 'P', 'xpore': 'X', 'epinano': 'p', 'nanodoc': 'h', 'differr': '*',
}

TOOL_DISPLAY_NAMES = {
    'jacusa2': 'JACUSA2',
    'drummer': 'DRUMMER',
    'eligos': 'ELIGOS2',
    'tombo': 'Tombo',
    'yanocomp': 'Yanocomp',
    'nanocompore': 'Nanocompore',
    'xpore': 'xPore',
    'epinano': 'EpiNano',
    'nanodoc': 'nanoDoc',
    'differr': 'DiffErr',
}

REF_LABELS = {'16s_88_rrsE': '16S rRNA', '23s_78_rrlB': '23S rRNA'}
PREVALENCE = {'16s_88_rrsE': 11 / 1542, '23s_78_rrlB': 25 / 2904}




### Shared Helpers
These small helpers keep file loading, coverage axes, figure saving, and repeated annotation logic consistent.


In [ ]:
# -----------------------------------------------------------------------------
# input/output configuration
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'rnamod-tool-benchmark-code':
    PROJECT_ROOT = PROJECT_ROOT.parent

NOTEBOOK_DIR = PROJECT_ROOT / 'rnamod-tool-benchmark-code'
if not NOTEBOOK_DIR.exists():
    raise FileNotFoundError(
        f'Could not locate the rnamod-tool-benchmark-code directory from {Path.cwd().resolve()}.'
    )

# Set these two paths if the collated rerun folder or figure output folder moves.
RUN_DIR = PROJECT_ROOT / 'rnamod-tool-benchmark-code'
FIG_DIR = NOTEBOOK_DIR / 'metric_figures'
GT_HEATMAP_SCOPE = '1000x'
GT_RECOVERY_STRICT_VALIDATION = True

# Reference geometry and evaluation scope
PADDING_NT = 200                                            # nt padding on rRNA references
N_REPLICATES = 3
REFERENCES = ['16s_88_rrsE', '23s_78_rrlB']
GT_SITE_COUNTS = {'16s_88_rrsE': 11, '23s_78_rrlB': 25}
GT_REFERENCE_LENGTHS = {'16s_88_rrsE': 1942, '23s_78_rrlB': 3304}
GT_EVAL_REGIONS     = {'16s_88_rrsE': (201, 1742), '23s_78_rrlB': (201, 3104)}
LABEL_COVS = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 500, 1000, 1200]

FIGURE_FILENAMES = {
    'nocall_summary_table': 'table_nocall_summary.csv',
    'pr_operating_points_replicates_table': 'supp_table_pr_operating_points_1000x_replicates.csv',
    'pr_operating_points_summary_table': 'supp_table_pr_operating_points_1000x_summary.csv',
    'gt_recovery_replicate_table': 'supp_table_gt_recovery_1000x_replicates.csv',
    'gt_recovery_consensus_table': 'supp_table_gt_recovery_1000x_consensus.csv',
}

if not RUN_DIR.exists():
    raise FileNotFoundError(
        f'Required collated rerun folder not found: {RUN_DIR}\n'
        'Update RUN_DIR at the top of the notebook and rerun.'
    )

FIG_DIR.mkdir(parents=True, exist_ok=True)


def require_run_file(filename):
    """Return a required CSV from the configured rerun directory."""
    path = RUN_DIR / filename
    if not path.exists():
        raise FileNotFoundError(
            f'Required file not found: {path}\n'
            'Update RUN_DIR at the top of the notebook and rerun.'
        )
    return path


def build_coverage_axis(df, coverage_col='coverage_label'):
    """Return ordered coverage labels and a lookup for evenly spaced x positions."""
    coverage_labels = list(df[coverage_col].cat.categories)
    x_lookup = {label: i for i, label in enumerate(coverage_labels)}
    return coverage_labels, x_lookup


def _cov_display(label):
    """Map coverage labels to display names (1200x -> Full)."""
    return 'Full' if str(label).startswith('1200') else str(label)


def save_figure_pair(fig, filename):
    """Save each figure as both PNG and SVG using the existing save settings."""
    fig.savefig(FIG_DIR / f'{filename}.png', dpi=300, bbox_inches='tight', facecolor='white')
    fig.savefig(FIG_DIR / f'{filename}.svg', bbox_inches='tight', facecolor='white')


def get_tool_style(tool):
    """Central lookup for tool color, marker, and display label."""
    return TOOL_COLORS[tool], TOOL_MARKERS[tool], TOOL_DISPLAY_NAMES.get(tool, tool)


def stagger_annotation_y(target_y, placed_y, min_gap, step):
    """Nudge annotation labels so nearby values do not overlap."""
    y = target_y
    for py in placed_y:
        if abs(y - py) < min_gap:
            y = py - step
    placed_y.append(y)
    return y


print(f'RUN_DIR: ./{RUN_DIR.relative_to(PROJECT_ROOT)}')
print(f'FIG_DIR: ./{FIG_DIR.relative_to(PROJECT_ROOT)}')

## Cell 2: Load and Validate Data

The references include `±200 nt` flanks for tool execution, but every benchmark metric in this notebook is evaluated only on the biological rRNA interval. Coordinates therefore remain in padded space while the expected evaluation universes are `1542` positions for 16S and `2904` positions for 23S.


In [ ]:

# Load data
metrics_csv = require_run_file('metrics_summary_long.csv')
df = pd.read_csv(metrics_csv)

# Coverage labels in numeric order (5x to 1000x)
coverage_labels = sorted(
    df['coverage_label'].dropna().astype(str).str.strip().unique(),
    key=lambda x: int(x.rstrip('xX'))
)

# Treat coverage as ordered categorical for consistent plotting order.
df['coverage_label'] = pd.Categorical(
    df['coverage_label'].astype(str).str.strip(),
    categories=coverage_labels,
    ordered=True,
)

# Parse coverage to numeric and keep rows in a stable order.
df['coverage_num'] = df['coverage_label'].str.replace('x', '').astype(int)
df = df.sort_values(['coverage_label', 'reference', 'tool']).reset_index(drop=True)

# Alias retained for compatibility with existing print/debug lines.
coverage_order = coverage_labels

print(f"Dataset: {len(df)} rows, {len(df['tool'].unique())} tools, {len(df['coverage_label'].unique())} coverage levels")
print(f"Coverage order (categorical labels): {coverage_order}")


In [ ]:
# Confirm the loaded metrics were generated on trimmed rRNA spans, not padded references.
metrics_long_csv_check = require_run_file('metrics_long.csv')
universe_check_df = pd.read_csv(metrics_long_csv_check, usecols=['reference', 'n_universe']).dropna()
universe_check_df['n_universe'] = pd.to_numeric(universe_check_df['n_universe'], errors='coerce').astype('Int64')
expected_universe = {ref: end - start + 1 for ref, (start, end) in GT_EVAL_REGIONS.items()}
observed_universe = (
    universe_check_df.groupby('reference')['n_universe']
    .apply(lambda s: sorted({int(v) for v in s.dropna().tolist()}))
    .to_dict()
)
missing_refs = sorted(set(expected_universe) - set(observed_universe))
found_stale = {value for values in observed_universe.values() for value in values if value in set(GT_REFERENCE_LENGTHS.values())}
unexpected = {
    ref: values for ref, values in observed_universe.items()
    if ref in expected_universe and values != [expected_universe[ref]]
}
if missing_refs or unexpected or found_stale:
    raise ValueError(
        "Stale padded-universe metrics detected. "
        f"Expected trimmed universe sizes, found {observed_universe}."
    )
print(f"Universe integrity check passed: {observed_universe}")



### Critical Calculation: Prevalence-Aware AUPRC
The two transformed AUPRC metrics below answer slightly different questions:
- `AUPRC / prevalence` asks how many times better than random a tool performs.
- `(AUPRC - p) / (1 - p)` removes chance performance and puts references on a comparable scale.


In [ ]:

# Reference-specific prevalence is constant within each reference.
df['prevalence'] = df['reference'].map(PREVALENCE)

# Fold-over-random AUPRC.
df['auprc_fold_mean'] = df['auprc_mean'] / df['prevalence']
df['auprc_fold_std'] = df['auprc_std'] / df['prevalence']

# Chance-corrected normalized AUPRC.
den = 1 - df['prevalence']
df['auprc_norm_mean'] = (df['auprc_mean'] - df['prevalence']) / den
df['auprc_norm_std'] = df['auprc_std'] / den

# Tool list is derived from the loaded summary table.
valid_metric_tools = set(df.loc[df['auprc_mean'].notna() | df['auroc_mean'].notna(), 'tool'].astype(str))
tools_with_metrics = [t for t in TOOL_ORDER if t in valid_metric_tools]
df_metrics = df[df['tool'].isin(tools_with_metrics)].copy()

print(f"Tools with valid metrics: {tools_with_metrics}")


## Cell 3: Reusable Plotting Function

In [ ]:
def plot_metric_vs_coverage(
    df, metric_mean, metric_std, ylabel, filename,
    tools=None, ylim=None, baseline_hlines=None, baseline_label=None, baseline_style='--',
    uncertainty='errorbar'  # 'band' | 'errorbar' | 'none'
):
    if tools is None:
        tools = tools_with_metrics
    
    refs = REFERENCES
    fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
    plt.subplots_adjust(hspace=0.15, bottom=0.18)
    
    # Coverage is categorical labels (ordered in load cell).
    coverage_labels, x_lookup = build_coverage_axis(df)
    
    for ax, ref in zip(axes, refs):
        sub = df[df['reference'] == ref]
        
        for tool in tools:
            t = sub[sub['tool'] == tool].copy()
            if t.empty or t[metric_mean].isna().all():
                continue
            
            t['coverage_label'] = t['coverage_label'].astype(str).str.strip()
            t['x'] = t['coverage_label'].map(x_lookup)
            t = t.dropna(subset=['x']).sort_values('x')
            
            x = t['x'].to_numpy(dtype=float)
            y = t[metric_mean].to_numpy(dtype=float)
            
            color, marker, display_name = get_tool_style(tool)

            ax.plot(
                x, y,
                color=color,
                marker=marker,
                label=display_name,
                markeredgecolor='white',
                markeredgewidth=0.5,
                markersize=5,
                linewidth=1.8
            )
            
            # Uncertainty rendering
            if metric_std and metric_std in t.columns and uncertainty != 'none':
                yerr = t[metric_std].to_numpy(dtype=float)
                valid = np.isfinite(yerr)
                if valid.any():
                    if uncertainty == 'band':
                        ax.fill_between(
                            x[valid], (y - yerr)[valid], (y + yerr)[valid],
                            color=color, alpha=0.15
                        )
                    elif uncertainty == 'errorbar':
                        ax.errorbar(
                            x[valid], y[valid], yerr=yerr[valid],
                            fmt='none',
                            ecolor=color,
                            elinewidth=0.7,
                            capsize=1.3,
                            alpha=0.25,
                            zorder=1
                        )
                    else:
                        raise ValueError("uncertainty must be one of: 'band', 'errorbar', 'none'")

        if baseline_hlines and ref in baseline_hlines:
            bval = baseline_hlines[ref]
            ax.axhline(y=bval, color='gray', linestyle=baseline_style, linewidth=1.2, alpha=0.7, zorder=0)
            if baseline_label:
                ax.text(
                    0.01, bval, baseline_label,
                    transform=ax.get_yaxis_transform(),
                    va='bottom', ha='left', fontsize=8, color='gray', style='italic'
                )

        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_title(REF_LABELS[ref], fontsize=13, fontweight='bold', pad=10)
        ax.set_xticks(range(len(coverage_labels)))
        ax.set_xticklabels([_cov_display(c) for c in coverage_labels], rotation=0, ha='center', fontsize=8)

        if ylim is not None:
            ax.set_ylim(ylim)

        ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
        ax.set_axisbelow(True)
    
    axes[1].set_xlabel('Sequencing Coverage', fontsize=12)
    for i, panel in enumerate(['a', 'b']):
        axes[i].text(-0.12, 1.2, panel, transform=axes[i].transAxes,
                     fontsize=25, fontweight='bold', va='top', ha='left')
    
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', ncol=5,
               bbox_to_anchor=(0.5, 0.02), frameon=True, fontsize=9,
               columnspacing=1.5, handletextpad=0.5)
    
    save_figure_pair(fig, filename)
    plt.show()
    print(f"Saved: {filename}.png and {filename}.svg")
    return fig

## Cell 4: Supp figure S3 — AUROC vs Coverage - two panel

In [ ]:
# Supplementary Figure: AUROC two-panel view (all tools with valid AUROC metrics, top-4 highlighted per reference)
def plot_auroc_two_panel_supp(df, save_prefix='suppfigS3_auroc_two_panel_supp'):
    """Two-panel AUROC vs coverage for all tools, highlighting top-4."""
    refs = REFERENCES
    tools = [t for t in tools_with_metrics if t in set(df['tool'].astype(str))]

    # Use the categorical coverage order defined in the load cell.
    coverage_labels, x_lookup = build_coverage_axis(df)

    fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
    plt.subplots_adjust(hspace=0.22, bottom=0.24)

    for ax, ref in zip(axes, refs):
        sub_ref = df[df['reference'] == ref].copy()

        # Rank tools by mean AUROC within this reference; highlight top 4 in this panel
        top4 = (
            sub_ref.groupby('tool')['auroc_mean']
            .mean()
            .sort_values(ascending=False)
            .head(4)
            .index
            .tolist()
        )

        for tool in tools:
            t = sub_ref[sub_ref['tool'] == tool].copy()
            if t.empty or t['auroc_mean'].isna().all():
                continue

            # Map categorical labels to equal-spaced x positions
            t['coverage_label'] = t['coverage_label'].astype(str).str.strip()
            t['x'] = t['coverage_label'].map(x_lookup)
            t = t.dropna(subset=['x']).sort_values('x')

            x = t['x'].to_numpy(dtype=float)
            y = t['auroc_mean'].to_numpy(dtype=float)

            color, marker, display_name = get_tool_style(tool)

            is_top = tool in top4
            line_alpha = 0.95 if is_top else 0.82
            line_width = 2.4 if is_top else 1.5
            marker_size = 4.4 if is_top else 2.8
            z = 3 if is_top else 2

            # Main AUROC line
            ax.plot(
                x, y,
                color=color,
                marker=marker,
                markersize=marker_size,
                linewidth=line_width,
                alpha=line_alpha,
                markeredgecolor='white' if is_top else '#222222',
                markeredgewidth=0.6,
                label=display_name,
                zorder=z
            )

            # Error bars at each coverage point for the included tools
            if 'auroc_std' in t.columns:
                yerr = t['auroc_std'].to_numpy(dtype=float)
                valid = np.isfinite(yerr)
                if valid.any():
                    ax.errorbar(
                        x[valid], y[valid], yerr=yerr[valid],
                        fmt='none',
                        ecolor=color,
                        elinewidth=0.95 if is_top else 0.70,
                        capsize=1.8,
                        alpha=0.35 if is_top else 0.25,
                        zorder=1
                    )

        # Random baseline
        ax.axhline(0.5, color='gray', linestyle='--', linewidth=1.0, alpha=0.7)

        # Panel formatting
        ax.set_title(REF_LABELS[ref], fontsize=16, pad=8)
        ax.set_ylabel('Mean AUROC (± SD)', fontsize=16)
        ax.set_ylim(0.4, 1.0)
        ax.set_xlim(-0.5, len(coverage_labels) - 0.5)

        # Show all coverage labels on BOTH panels
        ax.set_xticks(np.arange(len(coverage_labels)))
        ax.set_xticklabels([_cov_display(c) for c in coverage_labels], rotation=0, ha='center', fontsize=12)
        ax.tick_params(axis='x', labelbottom=True)

        ax.grid(True, axis='y', alpha=0.30, linewidth=0.5)
        ax.grid(True, axis='x', alpha=0.12, linewidth=0.35)
        ax.set_axisbelow(True)

    # Panel labels
    axes[0].text(-0.12, 1.2, 'a', transform=axes[0].transAxes, fontsize=25, fontweight='bold', va='top', ha='left')
    axes[1].text(-0.12, 1.2, 'b', transform=axes[1].transAxes, fontsize=25, fontweight='bold', va='top', ha='left')

    # Shared x-axis title
    axes[1].set_xlabel('Coverage', fontsize=16)

    # Single global legend (bottom)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles, labels,
        loc='lower center',
        ncol=5,
        bbox_to_anchor=(0.5, 0.02),
        frameon=True,
        fontsize=16,
        columnspacing=1.4,
        handletextpad=0.5
    )

    save_figure_pair(fig, save_prefix)
    plt.show()
    print(f"Saved: {save_prefix}.png and {save_prefix}.svg")
    return fig


# Run supplementary 2-panel AUROC figure
suppfigS3_auroc_two_panel_supp = plot_auroc_two_panel_supp(df_metrics, save_prefix='suppfigS3_auroc_two_panel_supp')

## Cell 5: Figure 3 — AUROC vs Coverage - 3x3 faceted plot

In [ ]:
def plot_auroc_faceted(df, filename):
    """3×N faceted AUROC vs coverage, one panel per tool."""
    refs = REFERENCES
    ref_colors = {'16s_88_rrsE': '#0072B2', '23s_78_rrlB': '#D55E00'}

    # Reuse categorical order set in your load cell.
    coverage_labels, x_lookup = build_coverage_axis(df)

    # Build tick indices dynamically: every 4th + always include last (Full)
    tick_idx = list(range(0, len(coverage_labels), 4))
    if (len(coverage_labels) - 1) not in tick_idx:
        # Drop previous tick if too close to the final "Full" label
        if tick_idx and (len(coverage_labels) - 1) - tick_idx[-1] < 3:
            tick_idx.pop()
        tick_idx.append(len(coverage_labels) - 1)
    tick_labels = [_cov_display(coverage_labels[i]) for i in tick_idx]

    ncols = 3
    nrows = int(np.ceil(len(tools_with_metrics) / ncols))
    fig_height = nrows * 3.1 if nrows <= 3 else 9.3 + (nrows - 3) * 3.8
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, fig_height), sharex=True, sharey=True)
    axes = np.atleast_1d(axes).flatten()

    slot_indices = list(range(len(tools_with_metrics)))
    if ncols == 3 and len(tools_with_metrics) % ncols == 1 and len(tools_with_metrics) > ncols:
        slot_indices = list(range(len(tools_with_metrics) - 1))
        slot_indices.append(ncols * (nrows - 1) + 1)

    used_slots = set(slot_indices)
    for tool, slot_idx in zip(tools_with_metrics, slot_indices):
        ax = axes[slot_idx]
        endpoint_labels = []

        for ref in refs:
            t = df[(df['tool'] == tool) & (df['reference'] == ref)].copy()
            if t.empty or t['auroc_mean'].isna().all():
                continue

            t['coverage_label'] = t['coverage_label'].astype(str).str.strip()
            t['x'] = t['coverage_label'].map(x_lookup)
            t = t.dropna(subset=['x']).sort_values('x')

            x = t['x'].to_numpy(dtype=float)
            y = t['auroc_mean'].to_numpy(dtype=float)

            ax.plot(
                x, y,
                color=ref_colors[ref],
                marker='o',
                markersize=3.2,
                linewidth=1.5,
                label=REF_LABELS[ref]
            )

            if 'auroc_std' in t.columns:
                yerr = t['auroc_std'].to_numpy(dtype=float)
                valid = np.isfinite(yerr)
                if valid.any():
                    ax.fill_between(x[valid], (y - yerr)[valid], (y + yerr)[valid],
                                    color=ref_colors[ref], alpha=0.12)

            endpoint_labels.append({
                'x': x[-1],
                'y': y[-1],
                'color': ref_colors[ref],
                'label': f'{y[-1]:.2f}',
            })

        if len(endpoint_labels) == 2 and abs(endpoint_labels[0]['y'] - endpoint_labels[1]['y']) < 0.03:
            if endpoint_labels[0]['y'] >= endpoint_labels[1]['y']:
                endpoint_labels[0]['y_offset'] = 0.012
                endpoint_labels[1]['y_offset'] = -0.012
            else:
                endpoint_labels[0]['y_offset'] = -0.012
                endpoint_labels[1]['y_offset'] = 0.012
        else:
            for endpoint in endpoint_labels:
                endpoint['y_offset'] = 0.0

        for endpoint in endpoint_labels:
            ax.text(
                endpoint['x'] + 0.20,
                endpoint['y'] + endpoint['y_offset'],
                endpoint['label'],
                color=endpoint['color'],
                fontsize=10,
                ha='left',
                va='center'
            )

        ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
        ax.set_title(TOOL_DISPLAY_NAMES.get(tool, tool), fontsize=14, fontweight='bold')
        ax.set_ylim(0.4, 1.0)
        ax.set_xlim(-0.5, len(coverage_labels) - 0.1)

        ax.set_xticks(tick_idx)
        ax.set_xticklabels(tick_labels, rotation=0, ha='center', fontsize=14)
        ax.tick_params(axis='x', labelbottom=True)

        ax.grid(True, axis='y', alpha=0.3, linewidth=0.5)
        ax.grid(True, axis='x', alpha=0.12, linewidth=0.4)
        ax.set_axisbelow(True)

    for idx, ax in enumerate(axes):
        if idx not in used_slots:
            ax.set_visible(False)

    fig.text(0.5, 0.03, 'Coverage', ha='center', fontsize=16)
    fig.text(0.02, 0.5, 'Mean AUROC', va='center', rotation='vertical', fontsize=16)

    handles, labels = axes[slot_indices[0]].get_legend_handles_labels()
    fig.legend(
        handles, labels,
        loc='upper center',
        bbox_to_anchor=(0.5, 1.02),   # outside plot area
        ncol=5,
        frameon=True,
        fontsize=16
    )

    # Reserve space for the external legend
    plt.tight_layout(rect=[0.04, 0.05, 1.00, 0.90])

    save_figure_pair(fig, filename)
    plt.show()

    print(f"Saved: {filename}.png and {filename}.svg")
    return fig

fig3_auroc_faceted = plot_auroc_faceted(df_metrics, 'fig3_auroc_faceted')

## Cell 6: Figure 4 — AUPRC vs Coverage - two panel

In [ ]:
fig4 = plot_metric_vs_coverage(
    df=df_metrics,
    metric_mean='auprc_mean',
    metric_std='auprc_std',
    ylabel='Mean AUPRC ($\\pm$ SD)',
    filename='fig4_auprc_vs_coverage',
    ylim=(0, 0.6),
    baseline_hlines=PREVALENCE,
    baseline_label='Prevalence baseline'
)


## Cell 7: Supplementary figure S4 — fold over random AUPRC vs Coverage

In [ ]:
figS3 = plot_metric_vs_coverage(
    df=df_metrics,
    metric_mean='auprc_fold_mean',
    metric_std='auprc_fold_std',
    ylabel='AUPRC / Prevalence (± SD)',
    filename='suppfigS4_auprc_fold_vs_coverage',
    ylim=(0, 70),
    baseline_hlines={'16s_88_rrsE': 1.0, '23s_78_rrlB': 1.0},
    baseline_label='Random = 1× prevalence',
    uncertainty='errorbar'
)


In [ ]:
# Load window & lag metrics

WINDOW_CSV = require_run_file('window_metrics_summary_long.csv')
LAG_CSV    = require_run_file('lag_metrics_summary_long.csv')

def load_window_metrics(path):
    df = pd.read_csv(path)
    df['window_size'] = df['window_size'].astype(int)
    # Ordered coverage factor
    cov_order = sorted(df['coverage_label'].unique(),
                       key=lambda x: int(x.replace('x', '')))
    df['coverage_label'] = pd.Categorical(df['coverage_label'],
                                           categories=cov_order, ordered=True)
    return df

def load_lag_metrics(path):
    df = pd.read_csv(path)
    df['delta'] = df['delta'].astype(int)
    cov_order = sorted(df['coverage_label'].unique(),
                       key=lambda x: int(x.replace('x', '')))
    df['coverage_label'] = pd.Categorical(df['coverage_label'],
                                           categories=cov_order, ordered=True)
    return df

df_window = load_window_metrics(WINDOW_CSV)
df_lag    = load_lag_metrics(LAG_CSV)

print(f"Window data: {len(df_window)} rows")
print(f"Lag data:    {len(df_lag)} rows")
print(f"Window sizes: {sorted(df_window['window_size'].unique())}")
print(f"Lag deltas:   {sorted(df_lag['delta'].unique())}")


## Cell 8 - Build lag metrics dictionary

In [ ]:
# Build nested dict: tool -> ref -> delta -> {auprc, auprc_std, prevalence}
# Reusable for any coverage level

def build_lag_dict(df, coverage='1000x', scope='universe'):
    """Extract lag metrics into a nested dict for a given coverage level."""
    sub = df[(df['coverage_label'] == coverage) &
             (df['scope'] == scope) &
             (df['auprc_mean'].notna())]
    
    out = defaultdict(lambda: defaultdict(dict))
    for _, r in sub.iterrows():
        tool, ref, d = r['tool'], r['reference'], int(r['delta'])
        out[tool][ref][d] = {
            'auprc':      r['auprc_mean'],
            'auprc_std':  r['auprc_std'] if pd.notna(r['auprc_std']) else 0.0,
            'prevalence':  r['prevalence_mean'] if pd.notna(r['prevalence_mean']) else 0,
        }
    return out

lag_tools = sorted(
    df_lag.loc[df_lag['auprc_mean'].notna(), 'tool'].unique(),
    key=lambda t: TOOL_ORDER.index(t) if t in TOOL_ORDER else 99
)

ldict_1000x = build_lag_dict(df_lag, coverage='1000x')


## Cell 9 - Function for the lag metrics plot

The asymmetry index in panel C is computed as:
`mean(AUPRC at +1..+5) - mean(AUPRC at -5..-1)`
so positive values indicate stronger downstream (3′) signal.


In [ ]:
# Figure 5: Lag/Offset Analysis — Directional Signal Displacement
#   a) AUPRC vs directional offset δ (−10 to +10) per tool
#   b) Row-normalized AUPRC heatmap (tools × δ)
#   c) Asymmetry index (diverging bar) + peak offset (dot chart)

def plot_lag_analysis(ldict, tools=None, save_prefix='fig6_lag_analysis'):
    """
    Three-panel figure for directional lag/offset analysis.
    Unlike window analysis, prevalence is constant across δ,
    so AUPRC changes are pure signal displacement — no inflation confound.
    """
    REFS   = REFERENCES
    DELTAS = list(range(-10, 11))

    if tools is None:
        tools = [t for t in TOOL_ORDER if any(ref in ldict.get(t, {}) for ref in REFS)]

    fig = plt.figure(figsize=(16, 18))
    gs = gridspec.GridSpec(3, 2, height_ratios=[1.0, 0.85, 0.85],
                           hspace=0.38, wspace=0.25,
                           left=0.08, right=0.92, top=0.96, bottom=0.06)

    # Panel A: AUPRC vs δ (line plots)
    for col, ref in enumerate(REFS):
        ax = fig.add_subplot(gs[0, col])
        for tool in tools:
            if ref not in ldict.get(tool, {}): continue
            ds = sorted(ldict[tool][ref].keys())
            y    = [ldict[tool][ref][d]['auprc'] for d in ds]
            yerr = [ldict[tool][ref][d]['auprc_std'] for d in ds]
            ax.plot(ds, y, color=TOOL_COLORS[tool], marker=TOOL_MARKERS[tool],
                    markersize=4.5, lw=1.6, markeredgecolor='white', markeredgewidth=0.4,
                    label=TOOL_DISPLAY_NAMES.get(tool, tool), zorder=3)
            ax.fill_between(ds, np.array(y)-np.array(yerr), np.array(y)+np.array(yerr),
                            color=TOOL_COLORS[tool], alpha=0.06, zorder=1)

        # 5-mer context shading + center line
        ax.axvspan(-2.5, 2.5, facecolor='#f5f1e8', zorder=0)
        ax.axvline(x=0, color='#333333', lw=1.0, ls='-', alpha=0.4, zorder=1)

        # Direction labels
        ax.annotate("← 5′ direction", xy=(0.02, 0.97), xycoords='axes fraction',
                    fontsize=8, color='#888888', ha='left', va='top', style='italic')
        ax.annotate("3′ direction →", xy=(0.98, 0.97), xycoords='axes fraction',
                    fontsize=8, color='#888888', ha='right', va='top', style='italic')

        ax.set_xlabel('Offset δ (nt from true modification site)', fontsize=18)
        ax.set_ylabel('AUPRC' if col == 0 else '', fontsize=18)
        ax.set_title(REF_LABELS[ref], fontsize=12, fontweight='bold', pad=12)
        ax.set_xlim(-10.5, 10.5)
        ax.set_xticks(range(-10, 11, 2))
        ax.tick_params(axis='both', labelsize=18)
        ax.grid(axis='y', alpha=0.2, lw=0.5)
        if col == 1:
            ax.legend(fontsize=7.5, loc='upper right', framealpha=0.92,
                      edgecolor='#cccccc', handlelength=1.8, labelspacing=0.3, ncol=1)

    fig.text(0.025, 0.97, 'a', fontsize=25, fontweight='bold', va='top')
    fig.text(0.50, 0.97,
             'AUPRC at each directional offset — no prevalence confound '
             '(n_positive constant across δ)',
             fontsize=12, ha='center', va='top', style='italic', color='#666666')

    # Panel B: Row-normalized AUPRC heatmaps
    for col, ref in enumerate(REFS):
        ax = fig.add_subplot(gs[1, col])
        active = [t for t in tools if ref in ldict.get(t, {}) and 0 in ldict[t][ref]]
        mat = np.full((len(active), len(DELTAS)), np.nan)
        for i, tool in enumerate(active):
            for j, d in enumerate(DELTAS):
                if d in ldict[tool][ref]:
                    mat[i, j] = ldict[tool][ref][d]['auprc']

        # Row-normalize for comparable visualization
        mat_norm = mat.copy()
        for i in range(mat_norm.shape[0]):
            rmax = np.nanmax(mat_norm[i, :])
            if rmax > 0:
                mat_norm[i, :] /= rmax

        im = ax.imshow(mat_norm, aspect='auto', cmap='YlOrRd',
                       interpolation='nearest', vmin=0, vmax=1)

        # Mark δ=0 column and peak per tool
        d0_idx = DELTAS.index(0)
        ax.axvline(x=d0_idx, color='white', lw=1.5, ls='-', alpha=0.8, zorder=2)
        for i in range(mat_norm.shape[0]):
            row = mat_norm[i, :]
            if np.any(np.isfinite(row)):
                ax.plot(np.nanargmax(row), i, 'w*', markersize=10,
                        markeredgecolor='black', markeredgewidth=0.5, zorder=5)

        ax.set_xticks(range(0, len(DELTAS), 2))
        ax.set_xticklabels([str(d) for d in DELTAS[::2]], fontsize=18)
        ax.set_yticks(range(len(active)))
        ax.set_yticklabels([TOOL_DISPLAY_NAMES.get(t, t) for t in active], fontsize=18)
        ax.set_xlabel('Offset δ (nt)', fontsize=18)
        if col == 0: ax.set_ylabel('Tool', fontsize=18)
        ax.set_title(f'AUPRC (row-normalized) — {REF_LABELS[ref]}',
                     fontsize=12, fontweight='bold', pad=7)
        cbar = fig.colorbar(im, ax=ax, shrink=0.85, pad=0.02)
        cbar.set_label('AUPRC / tool max', fontsize=18)
        cbar.ax.tick_params(labelsize=18)

    fig.text(0.025, 0.645, 'b', fontsize=25, fontweight='bold', va='top')
    fig.text(0.50, 0.645,
             '★ = peak offset per tool; white line = true modification site (δ=0)',
             fontsize=12, ha='center', va='top', style='italic', color='#666666')

    # Panel C-left: Asymmetry index
    ax_cl = fig.add_subplot(gs[2, 0])
    x_pos = np.arange(len(tools))
    bar_w = 0.35

    for ref_idx, ref in enumerate(REFS):
        # Compare average downstream (+1..+5) versus upstream (-5..-1) performance.
        asyms = []
        for tool in tools:
            if ref in ldict.get(tool, {}) and 0 in ldict[tool][ref]:
                d = ldict[tool][ref]
                pos_mean = np.nanmean([d.get(dd, {}).get('auprc', np.nan)
                                       for dd in range(1, 6)])
                neg_mean = np.nanmean([d.get(dd, {}).get('auprc', np.nan)
                                       for dd in range(-5, 0)])
                asyms.append(pos_mean - neg_mean)
            else:
                asyms.append(0)

        offset = -bar_w / 2 + ref_idx * bar_w
        color = '#3C5488' if ref_idx == 0 else '#E64B35'
        ax_cl.barh(x_pos + offset, asyms, height=bar_w, color=color,
                   edgecolor='white', lw=0.5, alpha=0.80, zorder=3,
                   label=REF_LABELS[ref])

    ax_cl.axvline(x=0, color='black', lw=1.0, zorder=2)
    ax_cl.set_yticks(x_pos)
    ax_cl.set_yticklabels([TOOL_DISPLAY_NAMES.get(t, t) for t in tools], fontsize=18)
    ax_cl.set_xlabel('Asymmetry index\n'
                     '(mean AUPRC δ=+1..+5) − (mean AUPRC δ=−1..−5)', fontsize=18)
    ax_cl.set_title("5′ ← Directional bias → 3′",
                    fontsize=12, fontweight='bold', pad=8)
    ax_cl.tick_params(axis='x', labelsize=18)
    ax_cl.grid(axis='x', alpha=0.2, lw=0.5)
    ax_cl.legend(fontsize=10, loc='lower right', framealpha=0.92, edgecolor='#cccccc')
    xlim = ax_cl.get_xlim()
    ax_cl.text(xlim[0] * 0.85, -0.8, "5′ bias\n(signal detected\nupstream)",
               fontsize=12, color='#888888', ha='left', va='top', style='italic')
    ax_cl.text(xlim[1] * 0.85, -0.8, "3′ bias\n(signal detected\ndownstream)",
               fontsize=12, color='#888888', ha='right', va='top', style='italic')
    ax_cl.invert_yaxis()

    # Panel C-right: Peak offset strip chart
    ax_cr = fig.add_subplot(gs[2, 1])

    REF_CFG = {
        '16s_88_rrsE': {'offset': -0.12, 'color': '#3C5488'},
        '23s_78_rrlB': {'offset':  0.12, 'color': '#E64B35'},
    }
    for ref, cfg in REF_CFG.items():
        xoff, color = cfg['offset'], cfg['color']
        for i, tool in enumerate(tools):
            if ref not in ldict.get(tool, {}) or 0 not in ldict[tool][ref]:
                continue
            d = ldict[tool][ref]
            vals = {dd: d[dd]['auprc'] for dd in DELTAS if dd in d}
            if not vals: continue
            peak_d = max(vals, key=vals.get)

            ax_cr.scatter(peak_d, i + xoff, s=100, marker='o',
                          facecolors=color, edgecolors='white', linewidths=1.0, zorder=4)
            if peak_d != 0:
                ax_cr.text(peak_d + 0.4, i + xoff, f'δ={peak_d:+d}',
                           fontsize=12, color=color, va='center', ha='left',
                           fontweight='bold')

    ax_cr.axvline(x=0, color='#333333', lw=1.2, ls='-', alpha=0.5, zorder=1)
    ax_cr.axvspan(-2.5, 2.5, facecolor='#f5f1e8', edgecolor='none', alpha=0.5, zorder=0)
    ax_cr.set_yticks(range(len(tools)))
    ax_cr.set_yticklabels([TOOL_DISPLAY_NAMES.get(t, t) for t in tools], fontsize=18)
    ax_cr.set_xlabel('Peak offset δ (nt from true site)', fontsize=18)
    ax_cr.set_title('δ that maximizes AUPRC per tool',
                    fontsize=12, fontweight='bold', pad=8)
    ax_cr.set_xlim(-11, 11)
    ax_cr.set_xticks(range(-10, 11, 2))
    ax_cr.tick_params(axis='x', labelsize=18)
    ax_cr.grid(axis='x', alpha=0.2, lw=0.5)
    ax_cr.invert_yaxis()
    ax_cr.annotate("← 5′", xy=(-10, len(tools) - 0.3), fontsize=12,
                   color='#888888', ha='left', va='center', fontweight='bold')
    ax_cr.annotate("3′ →", xy=(10, len(tools) - 0.3), fontsize=12,
                   color='#888888', ha='right', va='center', fontweight='bold')

    legend_el = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#3C5488',
               markeredgecolor='white', markersize=10, label='16S rRNA'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#E64B35',
               markeredgecolor='white', markersize=10, label='23S rRNA'),
    ]
    ax_cr.legend(handles=legend_el, fontsize=10, loc='lower right',
                 framealpha=0.92, edgecolor='#cccccc')

    fig.text(0.025, 0.31, 'c', fontsize=25, fontweight='bold', va='top')

    # Save
    save_figure_pair(fig, save_prefix)
    plt.show()
    print(f"Saved: {save_prefix}.png and .svg")
    return fig

## Cell 10 - Lag analysis Plot using the function defined above - Figure 6 - this shows if signal is consistently shifting from actual modification positions

In [ ]:
# Generate Figure 6 — Lag Analysis at 1000x coverage

ldict_1000x = build_lag_dict(df_lag, coverage='1000x', scope='universe')
fig6 = plot_lag_analysis(ldict_1000x, tools=lag_tools,
                         save_prefix='fig6_lag_analysis')



## Cell 11 - Load and prepare data for no-call / Ground Truth recall data - for Figure 5

No-call rate tells us how often a tool stays silent at reference positions.
Ground-truth recall tells us how often known modified sites receive any score at all.
Together they explain why some reported-scope metrics look much stronger than universe-scope metrics.


In [ ]:

metrics_csv = require_run_file('metrics_summary_long.csv')
df_main = pd.read_csv(metrics_csv)

# Keep coverage ordering aligned with earlier figures.
cov_order = [f'{x}x' for x in sorted(df_main['coverage_label']
             .astype(str).str.replace('x','').astype(int).unique())]
df_main['coverage_label'] = pd.Categorical(
    df_main['coverage_label'].astype(str), categories=cov_order, ordered=True)


def build_nocall_data(df):
    """Extract no-call rate and ground truth recall for each tool/reference/coverage."""
    out = {}
    for tool in TOOL_ORDER:
        out[tool] = {}
        for ref in ['16s_88_rrsE', '23s_78_rrlB']:
            out[tool][ref] = {}
            sub = df[(df['tool'] == tool) & (df['reference'] == ref)]
            for _, r in sub.iterrows():
                cov = int(str(r['coverage_label']).replace('x', ''))
                nc = r['no_call_rate_mean']
                gt = r['gt_recall_raw_mean']
                if pd.notna(nc) and pd.notna(gt):
                    out[tool][ref][cov] = {'no_call': nc, 'gt_recall': gt}
    return out


nocall_data = build_nocall_data(df_main)


## Cell 12 - function to plot the data - Figure 5

In [ ]:
# Figure 5: No-call Burden and Ground-Truth Site Coverage

def plot_nocall_gt_recall(nocall_data, save_prefix='fig5_nocall_gt_recall'):
    """Two-panel figure: no-call rate and GT site recall vs coverage."""

    REFS = REFERENCES
    REF_TITLES = {'16s_88_rrsE': '16S rRNA (11 known modifications)',
                  '23s_78_rrlB': '23S rRNA (25 known modifications)'}
    
    # Collect all coverage labels across all tools/refs, sort numerically
    all_covs = set()
    for tool in TOOL_ORDER:
        for ref in REFS:
            if ref in nocall_data[tool]:
                all_covs.update(nocall_data[tool][ref].keys())
    coverage_levels = sorted(int(str(c).replace('x','')) for c in all_covs)
    x_lookup = {cov: i for i, cov in enumerate(coverage_levels)}
    
    # ── X-axis: categorical spacing, show ticks only at key coverages ──
    key_positions = [x_lookup[c] for c in LABEL_COVS if c in x_lookup]
    key_labels    = [_cov_display(f'{c}x') for c in LABEL_COVS if c in x_lookup]
    # Drop ticks too close to their right neighbour (keeps rightmost / "Full")
    _pos, _lbl = [], []
    for i, (p, l) in enumerate(zip(key_positions, key_labels)):
        if i + 1 < len(key_positions) and key_positions[i + 1] - p < 3:
            continue
        _pos.append(p)
        _lbl.append(l)
    key_positions, key_labels = _pos, _lbl
    
    fig = plt.figure(figsize=(16, 11))
    gs = gridspec.GridSpec(2, 2, hspace=0.35, wspace=0.22,
                           left=0.07, right=0.88, top=0.93, bottom=0.08)

    # PANEL A: No-call rate vs coverage
    for col, ref in enumerate(REFS):
        ax = fig.add_subplot(gs[0, col])

        for tool in TOOL_ORDER:
            if ref not in nocall_data[tool] or not nocall_data[tool][ref]:
                continue
            covs_sorted = sorted(int(str(c).replace('x','')) for c in nocall_data[tool][ref].keys())
            x = [x_lookup[c] for c in covs_sorted]
            y = [nocall_data[tool][ref][c]['no_call'] for c in covs_sorted]
            
            ax.plot(x, y,
                    color=TOOL_COLORS[tool], marker=TOOL_MARKERS[tool],
                    markersize=5, lw=1.8, markeredgecolor='white',
                    markeredgewidth=0.5,
                    label=TOOL_DISPLAY_NAMES.get(tool, tool), zorder=3)

        # 50% reference line
        ax.axhline(y=0.5, color='#999999', lw=1.0, ls='--', alpha=0.5, zorder=1)
        if col == 1:
            ax.text(0, 0.52, '50%', fontsize=10, color='#999999',
                    va='bottom', ha='left', style='italic')

        # ── X-axis: categorical spacing with explicit "5x", "10x" style labels ──
        ax.set_xticks(key_positions)
        ax.set_xticklabels(key_labels, rotation=0, ha='center', fontsize=10)
        ax.set_xticks([], minor=True)
        
        ax.set_xlim(-0.5, len(coverage_levels) - 0.5)


        ax.set_ylim(-0.03, 1.03)
        ax.set_xlabel('Sequencing coverage', fontsize=10.5)
        ax.set_ylabel('No-call rate\n(fraction of positions with no score)'
                      if col == 0 else '', fontsize=10.5)
        ax.set_title(REF_TITLES[ref], fontsize=12, fontweight='bold', pad=8)
        ax.grid(axis='y', alpha=0.2, lw=0.5)

        # ── Annotate tools with no-call > 30% at 1000x ──
        annotations = [(t, nocall_data[t][ref][1000]['no_call'])
                       for t in TOOL_ORDER
                       if ref in nocall_data[t] and 1000 in nocall_data[t][ref]
                       and nocall_data[t][ref][1000]['no_call'] > 0.3]
        annotations.sort(key=lambda x: -x[1])
        placed_y = []
        for tool, nc in annotations:
            target_y = stagger_annotation_y(nc, placed_y, min_gap=0.06, step=0.07)
            ax.annotate(f'{TOOL_DISPLAY_NAMES[tool]} ({nc:.0%})',
                        xy=(x_lookup[1000], nc), xytext=(x_lookup[1000] + 0.5, target_y),
                        fontsize=7.5, color=TOOL_COLORS[tool],
                        fontweight='bold', ha='left', va='center',
                        arrowprops=dict(arrowstyle='-',
                                        color=TOOL_COLORS[tool],
                                        lw=0.8, alpha=0.5))

    fig.text(0.025, 0.95, 'a', fontsize=25, fontweight='bold', va='top')

    # PANEL B: Ground truth recall vs coverage
    for col, ref in enumerate(REFS):
        ax = fig.add_subplot(gs[1, col])

        for tool in TOOL_ORDER:
            if ref not in nocall_data[tool] or not nocall_data[tool][ref]:
                continue
            covs_sorted = sorted(int(str(c).replace('x','')) for c in nocall_data[tool][ref].keys())
            x = [x_lookup[c] for c in covs_sorted]
            y = [nocall_data[tool][ref][c]['gt_recall'] for c in covs_sorted]
            ax.plot(x, y,
                    color=TOOL_COLORS[tool], marker=TOOL_MARKERS[tool],
                    markersize=5, lw=1.8, markeredgecolor='white',
                    markeredgewidth=0.5,
                    label=TOOL_DISPLAY_NAMES.get(tool, tool), zorder=3)

        # 100% reference line
        ax.axhline(y=1.0, color='#999999', lw=0.8, ls='-', alpha=0.3, zorder=1)

        # ── X-axis: categorical spacing with explicit "5x", "10x" style labels ──
        ax.set_xticks(key_positions)
        ax.set_xticklabels(key_labels, rotation=0, ha='center', fontsize=10)
        ax.set_xticks([], minor=True)
        
        ax.set_xlim(-0.5, len(coverage_levels) - 0.5)

        ax.set_ylim(-0.03, 1.08)
        ax.set_xlabel('Sequencing coverage', fontsize=10.5)
        ax.set_ylabel('Ground truth site recall\n'
                      '(fraction of known sites receiving any score)'
                      if col == 0 else '', fontsize=10.5)
        ax.set_title(REF_TITLES[ref], fontsize=12, fontweight='bold', pad=8)
        ax.grid(axis='y', alpha=0.2, lw=0.5)

        # ── Annotate tools with gt_recall < 90% at 1000x ──
        annotations = [(t, nocall_data[t][ref][1000]['gt_recall'])
                       for t in TOOL_ORDER
                       if ref in nocall_data[t] and 1000 in nocall_data[t][ref]
                       and nocall_data[t][ref][1000]['gt_recall'] < 0.9]
        annotations.sort(key=lambda x: -x[1])
        placed_y = []
        for tool, gt in annotations:
            target_y = stagger_annotation_y(gt, placed_y, min_gap=0.07, step=0.08)
            ax.annotate(f'{TOOL_DISPLAY_NAMES[tool]} ({gt:.0%})',
                        xy=(x_lookup[1000], gt), xytext=(x_lookup[1000] + 0.5, target_y),
                        fontsize=7.5, color=TOOL_COLORS[tool],
                        fontweight='bold', ha='left', va='center',
                        arrowprops=dict(arrowstyle='-',
                                        color=TOOL_COLORS[tool],
                                        lw=0.8, alpha=0.5))

    fig.text(0.025, 0.49, 'b', fontsize=25, fontweight='bold', va='top')
    
    # Shared legend
    legend_handles = [
        Line2D([0], [0], color=TOOL_COLORS[t], marker=TOOL_MARKERS[t],
            markersize=7, lw=1.8, markeredgecolor='white', markeredgewidth=0.5,
            label=TOOL_DISPLAY_NAMES[t])
        for t in TOOL_ORDER
    ]
    fig.legend(handles=legend_handles, loc='lower center',
            ncol=5, fontsize=12, framealpha=0.92, edgecolor='#cccccc',
            handlelength=2.2, columnspacing=1.8,
            bbox_to_anchor=(0.47, -0.02))
    

    # ── Save ──
    save_figure_pair(fig, save_prefix)
    plt.show()
    print(f"Saved: {save_prefix}.png and .svg")
    return fig

## Cell 13 - Plot the Figure 5 - no-call rate and gt-recall vs coverages

In [ ]:
fig5_nocall = plot_nocall_gt_recall(nocall_data,
                                   save_prefix='fig5_nocall_gt_recall')


## Cell 14 - Generate summary table for no-call and ground truth recall

Critical calculation in this table: `Inflation = AUPRC(reported scope) / AUPRC(universe scope)`.
This ratio is what ties no-call behavior directly to optimistic reported-only performance.


In [ ]:
# Summary table: quantify how no-call behavior inflates reported-scope AUPRC.

rows_out = []
for ref in ['16s_88_rrsE', '23s_78_rrlB']:
    for tool in TOOL_ORDER:
        row = df_main[(df_main['tool'] == tool) &
                      (df_main['reference'] == ref) &
                      (df_main['coverage_label'] == '1000x')]
        if row.empty:
            continue
        r = row.iloc[0]
        au = r['auprc_mean']
        ar = r['auprc_reported_mean']
        nc = r['no_call_rate_mean']
        gt = r['gt_recall_raw_mean']
        
        # Inflation ratio: how much stronger reported-scope looks than universe-scope.
        ratio = ar / au if pd.notna(au) and pd.notna(ar) and au > 0 else np.nan
        
        rows_out.append({
            'Tool': TOOL_DISPLAY_NAMES.get(tool, tool),
            'Reference': REF_LABELS[ref],
            'No-call rate': f'{nc:.1%}',
            'GT site recall': f'{gt:.1%}',
            'AUPRC (universe)': f'{au:.3f}',
            'AUPRC (reported)': f'{ar:.3f}',
            'Inflation': f'{ratio:.1f}×' if not np.isnan(ratio) else 'N/A',
        })

df_nocall_summary = pd.DataFrame(rows_out)
display(df_nocall_summary)

df_nocall_summary.to_csv(f'{FIG_DIR}/table_nocall_gtrecall_summary.csv', index=False)
print(f"Saved: table_nocall_gtrecall_summary.csv")


## Cell 15 - Operating-Point Performance at 1000x Coverage

The AUROC and AUPRC analyses above summarize how well each tool ranks modified sites across thresholds. The first figure below shows where each tool lands in precision-recall space at its F1-optimal operating point at 1000x coverage, while the second figure translates the same operating behavior into recall versus false-positive burden. These plots should be interpreted as benchmark operating-point comparisons rather than validated deployment defaults, because the thresholds were selected from the benchmark itself.


In [ ]:

# Replicate-level metrics are enough to add PR operating-point figures now.
PR_CURVE_POINTS_FILE = None


def load_metrics_long_for_pr():
    """Load replicate-level benchmark metrics for the 1000x PR trade-off figures."""
    metrics_long_csv = require_run_file('metrics_long.csv')
    pr_metrics = pd.read_csv(metrics_long_csv)
    pr_metrics = pr_metrics[
        (pr_metrics['coverage_label'] == '1000x') &
        (pr_metrics['metrics_valid'] == True) &
        (pr_metrics['reference'].isin(['16s_88_rrsE', '23s_78_rrlB']))
    ].copy()
    pr_metrics = pr_metrics[pr_metrics['tool'].isin(tools_with_metrics)].copy()
    pr_metrics['replicate'] = pr_metrics['replicate'].astype(str)
    return pr_metrics


pr_metrics_1000x = load_metrics_long_for_pr()
print(f"PR metrics loaded: {len(pr_metrics_1000x)} replicate rows")
print(f"References: {sorted(pr_metrics_1000x['reference'].unique())}")
print(f"Tools: {sorted(pr_metrics_1000x['tool'].unique())}")


## Cell 16 - The next cell exports the exact 1000x operating-point values used in the two figures below. One table preserves the replicate-level points used for plotting, and a second table summarizes each tool/reference combination as mean ± SD.


In [ ]:
# Export the 1000x operating-point values behind the two figures below.
expected_refs = {'16s_88_rrsE', '23s_78_rrlB'}
observed_refs = set(pr_metrics_1000x['reference'].astype(str).unique())
if set(pr_metrics_1000x['coverage_label'].astype(str).unique()) != {'1000x'}:
    raise ValueError('PR operating-point export expected only 1000x rows.')
if observed_refs != expected_refs:
    raise ValueError(f'PR operating-point export expected references {expected_refs}, found {observed_refs}.')

consistency = pr_metrics_1000x.groupby('reference')[['n_ground_truth', 'n_universe']].nunique()
if not ((consistency['n_ground_truth'] == 1) & (consistency['n_universe'] == 1)).all():
    raise ValueError('n_ground_truth and n_universe must be unique within each reference for PR export.')

pr_export_replicates = pr_metrics_1000x[
    [
        'reference', 'tool', 'replicate', 'coverage_label',
        'precision_optimal', 'recall_optimal', 'f1_optimal',
        'n_false_positives', 'n_true_positives', 'n_ground_truth', 'n_universe'
    ]
].copy()
pr_export_replicates['reference_label'] = pr_export_replicates['reference'].map(REF_LABELS)
pr_export_replicates['tool_label'] = pr_export_replicates['tool'].map(TOOL_DISPLAY_NAMES)
pr_export_replicates['prevalence'] = (
    pr_export_replicates['n_ground_truth'] / pr_export_replicates['n_universe']
)
pr_export_replicates['gt_sites_recovered'] = pr_export_replicates['n_true_positives']
pr_export_replicates['false_positive_burden'] = pr_export_replicates['n_false_positives']
pr_export_replicates = pr_export_replicates[
    [
        'reference', 'reference_label', 'tool', 'tool_label', 'replicate', 'coverage_label',
        'precision_optimal', 'recall_optimal', 'f1_optimal',
        'n_false_positives', 'n_true_positives', 'gt_sites_recovered', 'false_positive_burden',
        'n_ground_truth', 'n_universe', 'prevalence'
    ]
].sort_values(['reference', 'tool', 'replicate']).reset_index(drop=True)

pr_export_summary = (
    pr_metrics_1000x
    .groupby(['reference', 'tool'], as_index=False)
    .agg(
        coverage_label=('coverage_label', 'first'),
        precision_optimal_mean=('precision_optimal', 'mean'),
        precision_optimal_sd=('precision_optimal', 'std'),
        recall_optimal_mean=('recall_optimal', 'mean'),
        recall_optimal_sd=('recall_optimal', 'std'),
        f1_optimal_mean=('f1_optimal', 'mean'),
        f1_optimal_sd=('f1_optimal', 'std'),
        n_false_positives_mean=('n_false_positives', 'mean'),
        n_false_positives_sd=('n_false_positives', 'std'),
        n_true_positives_mean=('n_true_positives', 'mean'),
        n_true_positives_sd=('n_true_positives', 'std'),
        n_ground_truth=('n_ground_truth', 'first'),
        n_universe=('n_universe', 'first'),
    )
)
pr_export_summary['reference_label'] = pr_export_summary['reference'].map(REF_LABELS)
pr_export_summary['tool_label'] = pr_export_summary['tool'].map(TOOL_DISPLAY_NAMES)
pr_export_summary['prevalence'] = pr_export_summary['n_ground_truth'] / pr_export_summary['n_universe']
pr_export_summary['precision_optimal_pct'] = pr_export_summary['precision_optimal_mean']
pr_export_summary['recall_optimal_pct'] = pr_export_summary['recall_optimal_mean']
pr_export_summary = pr_export_summary[
    [
        'reference', 'reference_label', 'tool', 'tool_label', 'coverage_label',
        'precision_optimal_mean', 'precision_optimal_sd',
        'recall_optimal_mean', 'recall_optimal_sd',
        'f1_optimal_mean', 'f1_optimal_sd',
        'n_false_positives_mean', 'n_false_positives_sd',
        'n_true_positives_mean', 'n_true_positives_sd',
        'n_ground_truth', 'n_universe', 'prevalence',
        'precision_optimal_pct', 'recall_optimal_pct'
    ]
].sort_values(['reference', 'tool']).reset_index(drop=True)

replicate_export_path = FIG_DIR / FIGURE_FILENAMES['pr_operating_points_replicates_table']
summary_export_path = FIG_DIR / FIGURE_FILENAMES['pr_operating_points_summary_table']
pr_export_replicates.to_csv(replicate_export_path, index=False)
pr_export_summary.to_csv(summary_export_path, index=False)

print(f'Saved: {replicate_export_path.name} ({len(pr_export_replicates)} rows)')
print(f'Saved: {summary_export_path.name} ({len(pr_export_summary)} rows)')
display(pr_export_summary)


## Cell 17
### What these calculations mean

A horizontal line at the prevalence shows the performance of a random classifier in PR space. Points far above that line are doing real discriminative work.

The iso-F1 contours are there for orientation, not ranking. They help the eye see whether a tool is gaining recall at a tolerable precision cost or just trading one weakness for another.

The recall-vs-false-positive panel is the same operating-point story in absolute terms. That is often easier to interpret when the practical question is how many extra candidates a validation experiment would have to chase.


In [ ]:


def draw_iso_f1(ax, f1_values=(0.2, 0.4, 0.6, 0.8)):
    """Add light iso-F1 guide lines to a precision-recall panel."""
    for f1 in f1_values:
        recall = np.linspace(0.01, 1.0, 300)
        denom = 2 * recall - f1
        precision = np.divide(f1 * recall, denom, out=np.full_like(recall, np.nan), where=denom > 0)
        mask = np.isfinite(precision) & (precision > 0) & (precision <= 1.0)
        ax.plot(recall[mask], precision[mask], '--', color='#D8D8D8', linewidth=0.6, zorder=0)
        valid_idx = np.where(mask)[0]
        if len(valid_idx) > 0:
            target_idx = valid_idx[np.argmin(np.abs(recall[valid_idx] - 0.92))]
            if 0.03 < precision[target_idx] < 0.97:
                ax.text(recall[target_idx] + 0.01, precision[target_idx], f'F1={f1:.1f}',
                        fontsize=6, color='#AAAAAA', va='center')


EXTRA_STYLE_HANDLES = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=6,
           alpha=0.35, markeredgecolor='none', label='Individual replicates'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=9,
           markeredgecolor='black', markeredgewidth=0.8, label='Mean ± SD'),
    Line2D([0], [0], linestyle='--', color='#D8D8D8', linewidth=0.8, label='Iso-F1 guides'),
    Line2D([0], [0], linestyle=':', color='#B5B5B5', linewidth=1.2, label='Prevalence baseline'),
]


def build_tool_legend_handles(tools):
    """Build legend handles for the standard tool set."""
    handles = []
    for tool in tools:
        color, marker, display_name = get_tool_style(tool)
        handles.append(
            Line2D([0], [0], marker=marker, color='w', markerfacecolor=color,
                   markeredgecolor='black', markeredgewidth=0.5, markersize=8,
                   label=display_name)
        )
    return handles


def plot_pr_operating_points_1000x(pr_metrics, save_prefix='fig7_pr_operating_points'):
    """Precision-recall scatter at 1000x coverage for each reference."""
    refs = ['16s_88_rrsE', '23s_78_rrlB']
    tools = [t for t in TOOL_ORDER if t in pr_metrics['tool'].unique()]
    fig, axes = plt.subplots(1, 2, figsize=(14, 6.5), sharey=True)
    plt.subplots_adjust(wspace=0.15, bottom=0.24)

    for idx, (ax, ref) in enumerate(zip(axes, refs)):
        ref_data = pr_metrics[pr_metrics['reference'] == ref].copy()
        draw_iso_f1(ax)

        if not ref_data.empty:
            prevalence = ref_data['n_ground_truth'].iloc[0] / ref_data['n_universe'].iloc[0]
            ax.axhline(prevalence, color='#B5B5B5', linewidth=1.2, linestyle=':', zorder=0)

        for tool in tools:
            tool_data = ref_data[ref_data['tool'] == tool].copy()
            if tool_data.empty:
                continue

            color, marker, _ = get_tool_style(tool)
            for _, row in tool_data.iterrows():
                ax.scatter(row['recall_optimal'], row['precision_optimal'], c=color, marker=marker,
                           s=50, alpha=0.35, edgecolors='none', zorder=3)

            mean_rec = tool_data['recall_optimal'].mean()
            mean_prec = tool_data['precision_optimal'].mean()
            rec_sd = tool_data['recall_optimal'].std()
            prec_sd = tool_data['precision_optimal'].std()

            ax.scatter(mean_rec, mean_prec, c=color, marker=marker, s=180,
                       edgecolors='black', linewidths=0.8, zorder=5)
            if pd.notna(rec_sd) and pd.notna(prec_sd):
                ax.errorbar(mean_rec, mean_prec, xerr=rec_sd, yerr=prec_sd,
                            fmt='none', ecolor=color, alpha=0.5, capsize=3,
                            linewidth=1.2, zorder=4)

        n_gt = int(ref_data['n_ground_truth'].iloc[0]) if not ref_data.empty else 0
        n_universe = int(ref_data['n_universe'].iloc[0]) if not ref_data.empty else 0
        ax.set_title(f"{REF_LABELS[ref]}\n({n_gt} GT sites / {n_universe} positions)", fontsize=12, fontweight='bold', pad=10)
        ax.set_xlabel('Recall', fontsize=11)
        if idx == 0:
            ax.set_ylabel('Precision', fontsize=11)
        ax.set_xlim(-0.02, 1.02)
        ax.set_ylim(-0.02, 1.02)
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.15)
        ax.set_axisbelow(True)
        ax.text(-0.34, 1.2, f"{chr(97 + idx)}", transform=ax.transAxes,
                fontsize=25, fontweight='bold', va='top', ha='left')

    legend_handles = build_tool_legend_handles(tools) + EXTRA_STYLE_HANDLES
    fig.legend(legend_handles, [h.get_label() for h in legend_handles], loc='lower center',
               ncol=4, bbox_to_anchor=(0.5, 0.02), frameon=True, fontsize=9,
               columnspacing=1.3, handletextpad=0.5)
    save_figure_pair(fig, save_prefix)
    plt.show()
    print(f"Saved: {save_prefix}.png and {save_prefix}.svg")
    return fig


def plot_recall_vs_fp_1000x(pr_metrics, save_prefix='fig8_recall_vs_fp'):
    """Recall vs false-positive count at 1000x for each reference."""
    refs = ['16s_88_rrsE', '23s_78_rrlB']
    tools = [t for t in TOOL_ORDER if t in pr_metrics['tool'].unique()]
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=False)
    plt.subplots_adjust(wspace=0.20, bottom=0.24)

    for idx, (ax, ref) in enumerate(zip(axes, refs)):
        ref_data = pr_metrics[pr_metrics['reference'] == ref].copy()

        for tool in tools:
            tool_data = ref_data[ref_data['tool'] == tool].copy()
            if tool_data.empty:
                continue

            color, marker, _ = get_tool_style(tool)
            for _, row in tool_data.iterrows():
                ax.scatter(row['n_false_positives'], row['recall_optimal'], c=color, marker=marker,
                           s=50, alpha=0.35, edgecolors='none', zorder=3)

            mean_fp = tool_data['n_false_positives'].mean()
            mean_rec = tool_data['recall_optimal'].mean()
            fp_sd = tool_data['n_false_positives'].std()
            rec_sd = tool_data['recall_optimal'].std()

            ax.scatter(mean_fp, mean_rec, c=color, marker=marker, s=180,
                       edgecolors='black', linewidths=0.8, zorder=5)
            if pd.notna(fp_sd) and pd.notna(rec_sd):
                ax.errorbar(mean_fp, mean_rec, xerr=fp_sd, yerr=rec_sd,
                            fmt='none', ecolor=color, alpha=0.5, capsize=3,
                            linewidth=1.2, zorder=4)

        n_gt = int(ref_data['n_ground_truth'].iloc[0]) if not ref_data.empty else 0
        n_universe = int(ref_data['n_universe'].iloc[0]) if not ref_data.empty else 0
        ax.set_title(f"{REF_LABELS[ref]}\n({n_gt} GT sites / {n_universe} positions)", fontsize=12, fontweight='bold', pad=10)
        ax.set_xlabel('False positives at F1-optimal threshold', fontsize=11)
        ax.set_ylabel('Recall', fontsize=11)
        ax.set_ylim(-0.02, 1.02)

        max_fp = ref_data['n_false_positives'].max() if not ref_data.empty else np.nan
        if pd.notna(max_fp) and max_fp > 50:
            ax.set_xscale('symlog', linthresh=1)
            ax.set_xlim(-0.5, max_fp * 1.5)
        else:
            ax.set_xlim(-0.5, max(5, float(max_fp) * 1.25 if pd.notna(max_fp) else 5))

        ax.grid(True, alpha=0.15)
        ax.set_axisbelow(True)
        ax.text(-0.34, 1.2, f"{chr(97 + idx)}", transform=ax.transAxes,
                fontsize=25, fontweight='bold', va='top', ha='left')
        ax.annotate('ideal corner', xy=(0, 1), fontsize=8, color='#888888',
                    style='italic', ha='left', va='top')

    legend_handles = build_tool_legend_handles(tools) + EXTRA_STYLE_HANDLES[:2]
    fig.legend(legend_handles, [h.get_label() for h in legend_handles], loc='lower center',
               ncol=4, bbox_to_anchor=(0.5, 0.02), frameon=True, fontsize=9,
               columnspacing=1.3, handletextpad=0.5)
    save_figure_pair(fig, save_prefix)
    plt.show()
    print(f"Saved: {save_prefix}.png and {save_prefix}.svg")
    return fig


def load_pr_curve_points_optional(curve_file):
    """Load optional precomputed PR curve points produced by the analysis pipeline."""
    if not curve_file:
        print('Skipping full PR curves: PR_CURVE_POINTS_FILE is not set.')
        return None

    curve_path = Path(curve_file)
    if not curve_path.exists():
        print(f'Skipping full PR curves: file not found -> {curve_path}')
        return None

    curve_df = pd.read_csv(curve_path)
    required = {'reference', 'tool', 'replicate', 'scope', 'index', 'precision', 'recall', 'threshold'}
    missing = required.difference(curve_df.columns)
    if missing:
        raise ValueError(f'Missing required PR curve columns: {sorted(missing)}')

    curve_df = curve_df[curve_df['reference'].isin(['16s_88_rrsE', '23s_78_rrlB'])].copy()
    curve_df = curve_df[curve_df['tool'].isin(tools_with_metrics)].copy()
    curve_df['replicate'] = curve_df['replicate'].astype(str)
    return curve_df


def plot_full_pr_curves_optional(curve_df, pr_metrics, save_prefix='pr_curves_full'):
    """Plot full PR curves if precomputed curve points are available."""
    if curve_df is None or curve_df.empty:
        print('Skipping full PR curves: no curve data available.')
        return None

    refs = ['16s_88_rrsE', '23s_78_rrlB']
    tools = [t for t in TOOL_ORDER if t in curve_df['tool'].unique()]
    common_recall = np.linspace(0, 1, 500)
    fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharey=True)
    plt.subplots_adjust(wspace=0.15, bottom=0.24)

    for idx, (ax, ref) in enumerate(zip(axes, refs)):
        ref_curves = curve_df[(curve_df['reference'] == ref) & (curve_df['scope'] == 'universe')].copy()
        ref_metrics = pr_metrics[pr_metrics['reference'] == ref].copy()
        draw_iso_f1(ax)

        if not ref_metrics.empty:
            prevalence = ref_metrics['n_ground_truth'].iloc[0] / ref_metrics['n_universe'].iloc[0]
            ax.axhline(prevalence, color='#B5B5B5', linewidth=1.2, linestyle=':', zorder=0)

        for tool in tools:
            tool_curves = ref_curves[ref_curves['tool'] == tool].copy()
            tool_metrics = ref_metrics[ref_metrics['tool'] == tool].copy()
            if tool_curves.empty or tool_metrics.empty:
                continue

            color, marker, _ = get_tool_style(tool)
            interpolated = []
            for rep in sorted(tool_curves['replicate'].unique()):
                rep_curve = tool_curves[tool_curves['replicate'] == rep].copy()
                rep_curve = rep_curve.sort_values(['recall', 'index'])
                rep_curve = rep_curve.groupby('recall', as_index=False)['precision'].max()
                if len(rep_curve) < 2:
                    continue
                ax.plot(rep_curve['recall'], rep_curve['precision'], color=color, linewidth=0.5,
                        alpha=0.25, zorder=1)
                interpolated.append(np.interp(common_recall, rep_curve['recall'], rep_curve['precision']))

            if not interpolated:
                continue

            mean_precision = np.mean(interpolated, axis=0)
            sd_precision = np.std(interpolated, axis=0)
            ax.plot(common_recall, mean_precision, color=color, linewidth=2.2, zorder=2)
            ax.fill_between(common_recall,
                            np.clip(mean_precision - sd_precision, 0, 1),
                            np.clip(mean_precision + sd_precision, 0, 1),
                            color=color, alpha=0.08, zorder=1)

            mean_rec = tool_metrics['recall_optimal'].mean()
            mean_prec = tool_metrics['precision_optimal'].mean()
            ax.scatter(mean_rec, mean_prec, c=color, marker=marker, s=140,
                       edgecolors='black', linewidths=0.8, zorder=5)

            for target_recall in (0.8, 0.9):
                if mean_rec >= target_recall or common_recall.max() >= target_recall:
                    target_precision = np.interp(target_recall, common_recall, mean_precision)
                    if np.isfinite(target_precision):
                        ax.scatter(target_recall, target_precision, marker='+', s=80,
                                   c=color, linewidths=2, zorder=5)

        n_gt = int(ref_metrics['n_ground_truth'].iloc[0]) if not ref_metrics.empty else 0
        n_universe = int(ref_metrics['n_universe'].iloc[0]) if not ref_metrics.empty else 0
        ax.set_title(f"{REF_LABELS[ref]}\n({n_gt} GT sites / {n_universe} positions)", fontsize=12, fontweight='bold', pad=10)
        ax.set_xlabel('Recall', fontsize=11)
        if idx == 0:
            ax.set_ylabel('Precision', fontsize=11)
        ax.set_xlim(-0.02, 1.02)
        ax.set_ylim(-0.02, 1.02)
        ax.grid(True, alpha=0.15)
        ax.set_axisbelow(True)
        ax.text(-0.34, 1.2, f"{chr(97 + idx)}", transform=ax.transAxes,
                fontsize=25, fontweight='bold', va='top', ha='left')

    legend_handles = build_tool_legend_handles(tools) + [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=8,
               markeredgecolor='black', markeredgewidth=0.8, label='F1-optimal point'),
        Line2D([0], [0], marker='+', color='gray', markersize=8, linewidth=0,
               markeredgewidth=2, label='Recall target (0.8 / 0.9)'),
        Line2D([0], [0], linestyle='--', color='#D8D8D8', linewidth=0.8, label='Iso-F1 guides'),
        Line2D([0], [0], linestyle=':', color='#B5B5B5', linewidth=1.2, label='Prevalence baseline'),
    ]
    fig.legend(legend_handles, [h.get_label() for h in legend_handles], loc='lower center',
               ncol=4, bbox_to_anchor=(0.5, 0.02), frameon=True, fontsize=9,
               columnspacing=1.3, handletextpad=0.5)
    save_figure_pair(fig, save_prefix)
    plt.show()
    print(f"Saved: {save_prefix}.png and {save_prefix}.svg")
    return fig


## Cell 18 - Plotting figures 7 and 8

In [ ]:

fig7_pr_operating_points = plot_pr_operating_points_1000x(pr_metrics_1000x, save_prefix='fig7_pr_operating_points')
fig8_recall_vs_fp = plot_recall_vs_fp_1000x(pr_metrics_1000x, save_prefix='fig8_recall_vs_fp')


## Cell 20 - GT Site Recovery at 1000x Coverage

The sections above quantify ranking performance, operating behavior, and callable-site burden across tools. The heatmaps below ask a different question: which specific known GT sites are recovered consistently, which are scored but fall below the F1-optimal threshold, and which remain no-calls at 1000x coverage. The main figure summarizes cross-replicate recovery fractions per tool, and the supplementary figures retain replicate-level detail.

In [ ]:
# Construct GT annotations directly from the padded benchmark positions already used in this notebook.

def _load_padded_gt_positions(path):
    """Load padded GT position file and return position array."""
    positions = []
    for line in Path(path).read_text().splitlines():
        value = line.strip()
        if not value or value.startswith('#'):
            continue
        positions.append(int(value))
    return positions

GT_MODIFICATION_TYPES = {
    '16s_88_rrsE': ['Ψ', 'm7G', 'm2G', 'm5C', 'm2G', 'm4Cm', 'm5C', 'm3U', 'm2G', 'm62A', 'm62A'],
    '23s_78_rrlB': ['m1G', 'Ψ', 'm5U', 'Ψ', 'm6A', 'm2G', 'Ψ', 'm3Ψ', 'Ψ', 'm5U', 'm5C', 'm6A', 'm7G', 'Gm', 'm2G', 'hU', 'Ψ', 'Cm', 'oh5C', 'm2A', 'Ψ', 'Um', 'Ψ', 'Ψ', 'Ψ'],
}

GT_POSITION_FILES = {
    '16s_88_rrsE': PROJECT_ROOT / 'tool_individual_outputs' / '16s_positions.txt',
    '23s_78_rrlB': PROJECT_ROOT / 'tool_individual_outputs' / '23s_positions.txt',
}


def load_gt_annotation():
    """Build GT annotation DataFrame from position files."""
    rows = []
    for reference in ['16s_88_rrsE', '23s_78_rrlB']:
        positions = _load_padded_gt_positions(GT_POSITION_FILES[reference])
        modifications = GT_MODIFICATION_TYPES[reference]
        if len(positions) != len(modifications):
            raise ValueError(
                f'GT annotation length mismatch for {reference}: '
                f'{len(positions)} padded positions vs {len(modifications)} modification types.'
            )
        for position, mod_type in zip(positions, modifications):
            padded_position = int(position)
            native_position = padded_position - PADDING_NT # Convert back to native coordinates by removing the PADDING_NT nt padding at the 5' end of rRNA
            rows.append({
                'reference': reference,
                'reference_label': REF_LABELS[reference],
                'position': padded_position,
                'native_position': native_position,
                'modification_type': mod_type,
                'site_label': f'{native_position} | {mod_type}',
            })
    gt_annotation = pd.DataFrame(rows)
    return gt_annotation.sort_values(['reference', 'position']).reset_index(drop=True)


gt_annotation = load_gt_annotation()


## Cell 21 - GT recovery heatmaps across replicates - calculations and functions for the plot

In [ ]:
import sys

DOWNSTREAM_DIR = (PROJECT_ROOT / 'downstream_analysis').resolve()
if str(DOWNSTREAM_DIR) not in sys.path:
    sys.path.insert(0, str(DOWNSTREAM_DIR))

from parse_outputs import load_all_tool_outputs

import logging
logging.getLogger('parse_outputs').setLevel(logging.WARNING)
from position_standardization import (
    build_metric_ready_table,
    canonicalize_tool_references,
    standardize_to_reference_universe,
)

GT_REFERENCE_CATALOG = pd.DataFrame([
    {'target': '16s', 'reference_id': '16s_88_rrsE'},
    {'target': '23s', 'reference_id': '23s_78_rrlB'},
])
REFERENCE_TARGET_NAMES = {'16s_88_rrsE': '16s', '23s_78_rrlB': '23s'}


def load_gt_recovery_inputs():
    """Load metrics and raw tool outputs needed for GT recovery analysis."""
    metrics_long = pd.read_csv(require_run_file('metrics_long.csv'))
    metrics_1000x = metrics_long[
        (metrics_long['coverage_label'] == GT_HEATMAP_SCOPE)
        & (metrics_long['metrics_valid'] == True)
        & (metrics_long['reference'].isin(REFERENCES))
        & (metrics_long['tool'].isin(TOOL_ORDER))
    ].copy()
    raw_outputs = load_all_tool_outputs(PROJECT_ROOT / 'tool_individual_outputs', tools=TOOL_ORDER)
    return metrics_1000x, raw_outputs, GT_REFERENCE_CATALOG.copy()


def build_gt_recovery_table_1000x(gt_annotation):
    """Classify each GT site as detected/below-threshold/no-call at 1000x."""
    metrics_1000x, raw_outputs, reference_catalog = load_gt_recovery_inputs()
    gt_positions = {
        ref: set(gt_annotation.loc[gt_annotation['reference'] == ref, 'position'].astype(int))
        for ref in gt_annotation['reference'].unique()
    }
    metric_ready_cache = {}

    for tool, raw_df in raw_outputs.items():
        canonical_df = canonicalize_tool_references(raw_df, reference_catalog)
        for reference in ['16s_88_rrsE', '23s_78_rrlB']:
            eval_start, eval_end = GT_EVAL_REGIONS[reference]
            standardized_df = standardize_to_reference_universe(
                canonical_df,
                reference,
                GT_REFERENCE_LENGTHS[reference],
                eval_start=eval_start,
                eval_end=eval_end,
            )
            metric_ready_cache[(tool, reference)] = build_metric_ready_table(
                standardized_df,
                ground_truth_positions=gt_positions[reference],
            )

    rows = []
    for _, metric_row in metrics_1000x.iterrows():
        tool = str(metric_row['tool'])
        reference = str(metric_row['reference'])
        replicate = str(metric_row['replicate'])
        threshold = float(metric_row['optimal_threshold'])

        metric_ready_df = metric_ready_cache[(tool, reference)]
        gt_rows = metric_ready_df[
            (metric_ready_df['reference'].astype(str) == reference)
            & (metric_ready_df['replicate'].astype(str) == replicate)
            & (metric_ready_df['label'].astype(int) == 1)
        ].copy()

        gt_rows = gt_rows.merge(
            gt_annotation[gt_annotation['reference'] == reference],
            on=['reference', 'position'],
            how='left',
            validate='one_to_one',
        )

        if gt_rows['modification_type'].isna().any():
            missing = gt_rows.loc[gt_rows['modification_type'].isna(), 'position'].astype(int).tolist()
            raise ValueError(f'Missing GT annotations for {reference} positions: {missing}')

        score_metric = pd.to_numeric(gt_rows['score_metric'], errors='coerce')
        is_reported = gt_rows['is_reported'].fillna(False).astype(bool)
        # Use an equality guard so threshold-edge sites parsed from text still match the benchmark counts.
        detected = is_reported & ((score_metric >= threshold) | np.isclose(score_metric, threshold, rtol=1e-9, atol=1e-9))
        below_threshold = is_reported & ~detected
        no_call = ~is_reported

        gt_rows['reference_label'] = REF_LABELS[reference]
        gt_rows['tool'] = tool
        gt_rows['tool_label'] = TOOL_DISPLAY_NAMES.get(tool, tool)
        gt_rows['coverage_label'] = GT_HEATMAP_SCOPE
        gt_rows['optimal_threshold'] = threshold
        gt_rows['score_metric'] = score_metric
        gt_rows['is_reported'] = is_reported
        gt_rows['detected'] = detected
        gt_rows['below_threshold'] = below_threshold
        gt_rows['no_call'] = no_call
        rows.append(
            gt_rows[[
                'reference', 'reference_label', 'position', 'native_position', 'modification_type', 'site_label',
                'tool', 'tool_label', 'replicate', 'coverage_label', 'score_metric',
                'optimal_threshold', 'is_reported', 'detected', 'below_threshold', 'no_call',
            ]]
        )

    gt_recovery_replicates = pd.concat(rows, ignore_index=True)
    return gt_recovery_replicates, metrics_1000x


def validate_gt_recovery_table(gt_recovery_replicates, metrics_1000x):
    """Check recovery table against metric-ready expected counts."""
    expected_site_counts = GT_SITE_COUNTS
    mismatches = []

    for _, metric_row in metrics_1000x.iterrows():
        tool = str(metric_row['tool'])
        reference = str(metric_row['reference'])
        replicate = str(metric_row['replicate'])
        sub = gt_recovery_replicates[
            (gt_recovery_replicates['tool'] == tool)
            & (gt_recovery_replicates['reference'] == reference)
            & (gt_recovery_replicates['replicate'].astype(str) == replicate)
        ].copy()

        expected_sites = expected_site_counts[reference]
        observed_sites = int(len(sub))
        if observed_sites != expected_sites:
            mismatches.append(
                f'{tool} {reference} {replicate}: expected {expected_sites} GT rows, found {observed_sites}'
            )

        observed_reported = int(sub['is_reported'].astype(bool).sum())
        expected_reported = int(metric_row['n_gt_reported'])
        if observed_reported != expected_reported:
            mismatches.append(
                f'{tool} {reference} {replicate}: n_gt_reported expected {expected_reported}, found {observed_reported}'
            )

        observed_detected = int(sub['detected'].astype(bool).sum())
        expected_detected = int(metric_row['n_true_positives'])
        if observed_detected != expected_detected:
            mismatches.append(
                f'{tool} {reference} {replicate}: n_true_positives expected {expected_detected}, found {observed_detected}'
            )

    if mismatches and GT_RECOVERY_STRICT_VALIDATION:
        joined = '\n'.join(mismatches)
        raise ValueError(f'GT recovery validation failed:\n{joined}')

    return mismatches


def build_gt_recovery_consensus_table(gt_recovery_replicates):
    """Aggregate replicate-level recovery into consensus fractions."""
    consensus = (
        gt_recovery_replicates
        .groupby(
            ['reference', 'reference_label', 'position', 'native_position', 'modification_type', 'site_label', 'tool', 'tool_label'],
            as_index=False,
        )
        .agg(
            n_detected=('detected', 'sum'),
            n_below_threshold=('below_threshold', 'sum'),
            n_no_call=('no_call', 'sum'),
        )
    )
    consensus['detected_fraction'] = consensus['n_detected'] / N_REPLICATES
    consensus['callable_fraction'] = (consensus['n_detected'] + consensus['n_below_threshold']) / N_REPLICATES
    return consensus


def get_gt_tool_order(consensus_df):
    """Return tools ordered by descending detected-fraction for heatmap rows."""
    tool_means = consensus_df.groupby('tool')['detected_fraction'].mean().to_dict()
    return sorted(
        [tool for tool in TOOL_ORDER if tool in set(consensus_df['tool'])],
        key=lambda tool: (-tool_means.get(tool, -1), TOOL_ORDER.index(tool)),
    )


def plot_gt_recovery_consensus_heatmaps(consensus_df, save_prefix='fig12_gt_recovery_consensus'):
    """Heatmap of consensus GT recovery states across tools."""
    tool_order = get_gt_tool_order(consensus_df)
    cmap = ListedColormap(['#F7FBFF', '#C6DBEF', '#6BAED6', '#08519C'])
    norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap.N)

    references = ['16s_88_rrsE', '23s_78_rrlB']
    row_counts = [GT_SITE_COUNTS[r] for r in references]
    fig, axes = plt.subplots(2, 1, figsize=(10.5, 11.8), gridspec_kw={'height_ratios': row_counts, 'hspace': 0.28})
    axes = np.atleast_1d(axes)

    for ax, reference in zip(axes, references):
        ref_sites = gt_annotation[gt_annotation['reference'] == reference].sort_values('position')
        ref_data = consensus_df[consensus_df['reference'] == reference].copy()
        matrix = pd.DataFrame(index=ref_sites['site_label'], columns=tool_order, dtype=float)
        for tool in tool_order:
            sub = ref_data[ref_data['tool'] == tool].set_index('site_label')
            matrix.loc[sub.index, tool] = sub['n_detected']
        matrix = matrix.fillna(0.0)

        im = ax.imshow(matrix.to_numpy(dtype=float), aspect='auto', cmap=cmap, norm=norm)
        ax.grid(False)
        ax.set_title(REF_LABELS[reference])
        ax.set_xticks(range(len(tool_order)))
        ax.set_xticklabels([TOOL_DISPLAY_NAMES.get(tool, tool) for tool in tool_order], rotation=45, ha='right')
        ax.set_yticks(range(len(ref_sites)))
        ax.set_yticklabels(ref_sites['site_label'])
        if reference == '23s_78_rrlB':
            ax.set_xlabel('Tool')

        ax.set_xticks(np.arange(-0.5, len(tool_order), 1), minor=True)
        ax.set_yticks(np.arange(-0.5, len(ref_sites), 1), minor=True)
        ax.grid(which='minor', color='white', linewidth=0.8)
        ax.tick_params(which='minor', bottom=False, left=False)

        for row_idx in range(matrix.shape[0]):
            for col_idx in range(matrix.shape[1]):
                value = int(matrix.iat[row_idx, col_idx])
                text_color = 'white' if value >= 2 else 'black'
                ax.text(col_idx, row_idx, f'{value}/3', ha='center', va='center', fontsize=9, color=text_color)

    fig.tight_layout()
    save_figure_pair(fig, save_prefix)
    return fig


def plot_gt_recovery_replicate_heatmap(gt_recovery_replicates, reference, save_prefix):
    """Per-replicate GT recovery state heatmap for one reference."""
    tool_order = get_gt_tool_order(build_gt_recovery_consensus_table(gt_recovery_replicates))
    ref_sites = gt_annotation[gt_annotation['reference'] == reference].sort_values('position')
    replicates = ['rep1', 'rep2', 'rep3']
    column_pairs = [(tool, rep) for tool in tool_order for rep in replicates]
    state_matrix = pd.DataFrame(index=ref_sites['site_label'], columns=pd.MultiIndex.from_tuples(column_pairs), dtype=float)

    ref_data = gt_recovery_replicates[gt_recovery_replicates['reference'] == reference].copy()
    for tool, rep in column_pairs:
        sub = ref_data[(ref_data['tool'] == tool) & (ref_data['replicate'].astype(str) == rep)].set_index('site_label')
        state = np.select(
            [sub.get('detected', False), sub.get('below_threshold', False), sub.get('no_call', False)],
            [2, 1, 0],
            default=np.nan,
        )
        if len(sub) > 0:
            state_matrix.loc[sub.index, (tool, rep)] = state
    state_matrix = state_matrix.fillna(0.0)

    cmap = ListedColormap(['#FFFFFF', '#D9D9D9', '#1F4E79'])
    norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], cmap.N)
    height = 5.6 if reference == '16s_88_rrsE' else 8.8
    fig, ax = plt.subplots(figsize=(14.5, height))
    ax.imshow(state_matrix.to_numpy(dtype=float), aspect='auto', cmap=cmap, norm=norm)
    ax.grid(False)

    ax.set_yticks(range(len(ref_sites)))
    ax.set_yticklabels(ref_sites['site_label'])
    ax.set_xticks(range(len(column_pairs)))
    ax.set_xticklabels(['r1', 'r2', 'r3'] * len(tool_order), rotation=90)
    ax.set_xlabel('Tool and replicate')

    ax.set_xticks(np.arange(-0.5, len(column_pairs), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(ref_sites), 1), minor=True)
    ax.grid(which='minor', color='#FFFFFF', linewidth=0.7)
    ax.tick_params(which='minor', bottom=False, left=False)

    for idx, tool in enumerate(tool_order):
        center = idx * 3 + 1
        ax.text(center, 1.02, TOOL_DISPLAY_NAMES.get(tool, tool), transform=ax.get_xaxis_transform(),
                ha='center', va='bottom', fontsize=9)
        if idx > 0:
            ax.axvline(idx * 3 - 0.5, color='black', linewidth=1.0)

    legend_handles = [
        Patch(facecolor='#1F4E79', edgecolor='black', label='Detected'),
        Patch(facecolor='#D9D9D9', edgecolor='black', label='Below threshold'),
        Patch(facecolor='#FFFFFF', edgecolor='black', label='No call'),
    ]
    ax.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, -0.08), ncol=3, frameon=False)
    fig.tight_layout()
    save_figure_pair(fig, save_prefix)
    return fig

## Cell 22 - GT recovery calculation outputs to tables

In [ ]:
# Reconstruct GT-site recovery states at 1000x from the existing downstream parser and metric-ready tables.
gt_recovery_replicates_1000x, gt_recovery_metrics_1000x = build_gt_recovery_table_1000x(gt_annotation)
validation_mismatches = validate_gt_recovery_table(gt_recovery_replicates_1000x, gt_recovery_metrics_1000x)
gt_recovery_consensus_1000x = build_gt_recovery_consensus_table(gt_recovery_replicates_1000x)

replicate_export_path = FIG_DIR / FIGURE_FILENAMES['gt_recovery_replicate_table']
consensus_export_path = FIG_DIR / FIGURE_FILENAMES['gt_recovery_consensus_table']

gt_recovery_replicates_1000x.sort_values(
    ['reference', 'position', 'tool', 'replicate']
).to_csv(replicate_export_path, index=False)

gt_recovery_consensus_1000x.sort_values(
    ['reference', 'position', 'tool']
).to_csv(consensus_export_path, index=False)

print(f'GT recovery: {len(gt_recovery_replicates_1000x)} replicate rows, '
      f'{len(gt_recovery_consensus_1000x)} consensus rows, '
      f'{len(validation_mismatches)} validation mismatches')


## Cell 23 - Figure 9 and Supplementary Figure S5 and S6 plotting

In [ ]:
fig9_gt_recovery_consensus = plot_gt_recovery_consensus_heatmaps(
    gt_recovery_consensus_1000x,
    save_prefix='fig9_gt_recovery_consensus',
)
figS5_gt_recovery_replicates_16s = plot_gt_recovery_replicate_heatmap(
    gt_recovery_replicates_1000x,
    '16s_88_rrsE',
    save_prefix='suppfigS5_gt_recovery_replicate_16s',
)
figS6_gt_recovery_replicates_23s = plot_gt_recovery_replicate_heatmap(
    gt_recovery_replicates_1000x,
    '23s_78_rrlB',
    save_prefix='suppfigS6_gt_recovery_replicate_23s',
)

## Cell 24 - Window analysis dictionary building

In [ ]:
# Build nested dict: tool -> ref -> window -> {auprc, auprc_std, prevalence, norm_auprc}
# Reusable for any coverage level

def build_window_dict(df, coverage='1000x', scope='reported'):
    """Extract window metrics into a nested dict for a given coverage level."""
    sub = df[(df['coverage_label'] == coverage) &
             (df['scope'] == scope) &
             (df['auprc_mean'].notna())]
    
    out = defaultdict(lambda: defaultdict(dict))
    for _, r in sub.iterrows():
        tool, ref, w = r['tool'], r['reference'], int(r['window_size'])
        prev  = r['prevalence_mean'] if pd.notna(r['prevalence_mean']) else 0
        auprc = r['auprc_mean']
        out[tool][ref][w] = {
            'auprc':      auprc,
            'auprc_std':  r['auprc_std'] if pd.notna(r['auprc_std']) else 0.0,
            'prevalence':  prev,
            'norm_auprc':  auprc / prev if prev > 0 else np.nan,
        }
    return out

wdict = build_window_dict(df_window, coverage='1000x')
print("Window dict built for 1000x coverage.")

## Cell 25 - Supplementary Figure S9 plotting

In [ ]:
# Supplementary Figure S9: Window Analysis

def plot_window_analysis(wdict, tools=None, save_prefix='suppfigS9_window_analysis'):
    """
    Three-panel figure for window relaxation analysis.
      a) AUPRC vs window size (raw line plots)
      b) ΔAUPRC heatmap (change from exact match, w=0)
      c-left) Prevalence vs window (inflation context)
      c-right) Dumbbell chart: AUPRC at w=0 vs peak AUPRC per tool
    """
    REFS    = REFERENCES
    WINDOWS = list(range(0, 11))

    if tools is None:
        tools = [t for t in TOOL_ORDER if any(ref in wdict.get(t, {}) for ref in REFS)]

    fig = plt.figure(figsize=(16, 18))
    gs = gridspec.GridSpec(3, 2, height_ratios=[1.0, 0.85, 0.85],
                           hspace=0.38, wspace=0.25,
                           left=0.08, right=0.92, top=0.96, bottom=0.06)

    # ── Panel a: AUPRC vs Window Size ─────────────────────────────────────
    for col, ref in enumerate(REFS):
        ax = fig.add_subplot(gs[0, col])
        for tool in tools:
            if ref not in wdict.get(tool, {}):
                continue
            ws = sorted(wdict[tool][ref].keys())
            y    = [wdict[tool][ref][w]['auprc'] for w in ws]
            yerr = [wdict[tool][ref][w]['auprc_std'] for w in ws]
            color, marker, display = get_tool_style(tool)
            ax.plot(ws, y, color=color, marker=marker,
                    markersize=4.5, lw=1.6, markeredgecolor='white',
                    markeredgewidth=0.4, label=display, zorder=3)
            ax.fill_between(ws, np.array(y) - np.array(yerr),
                            np.array(y) + np.array(yerr),
                            color=color, alpha=0.06, zorder=1)

        # 5-mer context shading (±2 nt)
        ax.axvspan(-0.3, 2, facecolor='#f5f1e8', zorder=0)
        ax.axvline(x=2, color='#c4b078', lw=0.8, ls=':', alpha=0.7, zorder=1)
        if col == 0:
            ax.annotate('5-mer\ncontext', xy=(0.85, 0.92),
                        xycoords=('data', 'axes fraction'),
                        fontsize=7.5, color='#8b7d5e', ha='center', va='top',
                        style='italic',
                        bbox=dict(boxstyle='round,pad=0.15', fc='#f5f1e8',
                                  ec='none', alpha=0.9))
        ax.set_xlabel('Window size (±nt from known site)', fontsize=18)
        ax.set_ylabel('AUPRC' if col == 0 else '', fontsize=18)
        ax.set_title(REF_LABELS[ref], fontsize=12, fontweight='bold', pad=12)
        ax.set_xlim(-0.3, 10.3)
        ax.set_ylim(-0.02, 1.02)
        ax.set_xticks(range(11))
        ax.tick_params(axis='both', labelsize=14)
        ax.grid(axis='y', alpha=0.2, lw=0.5)
        if col == 1:
            ax.legend(fontsize=7.5, loc='upper right', framealpha=0.92,
                      edgecolor='#cccccc', handlelength=1.8, labelspacing=0.3)

    fig.text(0.025, 0.97, 'a', fontsize=25, fontweight='bold', va='top')

    # ── Panel b: ΔAUPRC heatmap (change from exact match) ────────────────
    for col, ref in enumerate(REFS):
        ax = fig.add_subplot(gs[1, col])
        active = [t for t in tools
                  if ref in wdict.get(t, {}) and 0 in wdict[t][ref]]
        mat = np.full((len(active), len(WINDOWS)), np.nan)
        for i, tool in enumerate(active):
            for j, w in enumerate(WINDOWS):
                if w in wdict[tool][ref]:
                    mat[i, j] = wdict[tool][ref][w]['auprc']

        # Compute ΔAUPRC: change from w=0
        mat_delta = mat.copy()
        for i in range(mat_delta.shape[0]):
            baseline = mat_delta[i, 0]
            if np.isfinite(baseline):
                mat_delta[i, :] -= baseline

        vmax = np.nanmax(np.abs(mat_delta))
        im = ax.imshow(mat_delta, aspect='auto', cmap='RdBu_r',
                       interpolation='nearest', vmin=-vmax, vmax=vmax)

        # Mark w=0 column and peak |Δ| per tool
        w0_idx = WINDOWS.index(0)
        ax.axvline(x=w0_idx, color='black', lw=1.5, ls='-', alpha=0.4, zorder=2)
        for i in range(mat_delta.shape[0]):
            row = mat_delta[i, :]
            if np.any(np.isfinite(row)):
                peak_j = np.nanargmax(np.abs(row))
                ax.plot(peak_j, i, 'k*', markersize=10,
                        markeredgecolor='white', markeredgewidth=0.5, zorder=5)

        ax.set_xticks(range(len(WINDOWS)))
        ax.set_xticklabels([f'±{w}' for w in WINDOWS], fontsize=14)
        ax.set_yticks(range(len(active)))
        ax.set_yticklabels(
            [TOOL_DISPLAY_NAMES.get(t, t) for t in active], fontsize=14)
        ax.set_xlabel('Window size (±nt)', fontsize=18)
        if col == 0:
            ax.set_ylabel('Tool', fontsize=18)
        ax.set_title(f'ΔAUPRC from exact match — {REF_LABELS[ref]}',
                     fontsize=12, fontweight='bold', pad=7)
        cbar = fig.colorbar(im, ax=ax, shrink=0.85, pad=0.02)
        cbar.set_label('AUPRC(w) − AUPRC(w=0)', fontsize=14)
        cbar.ax.tick_params(labelsize=14)

    fig.text(0.025, 0.645, 'b', fontsize=25, fontweight='bold', va='top')
    fig.text(0.50, 0.645,
             '★ = largest absolute change; '
             'red = gain, blue = loss from exact match',
             fontsize=11, ha='center', va='top', style='italic', color='#666666')

    # ── Panel c-left: Prevalence vs window size ───────────────────────────
    ax_cl = fig.add_subplot(gs[2, 0])

    ref_colors = {'16s_88_rrsE': '#3C5488', '23s_78_rrlB': '#E64B35'}
    for ref in REFS:
        # Use prevalence from a representative tool (most share the same value)
        rep_tool = next(
            (t for t in tools
             if ref in wdict.get(t, {}) and 0 in wdict[t][ref]
             and wdict[t][ref][0]['prevalence'] < 0.02),
            tools[0])
        ws_vals = sorted(wdict[rep_tool][ref].keys())
        prev = [wdict[rep_tool][ref][w]['prevalence'] for w in ws_vals]
        ax_cl.plot(ws_vals, prev, 'o-', color=ref_colors[ref], lw=2,
                   markersize=5, label=REF_LABELS[ref], zorder=3)
        # Annotate fold-change
        if prev[0] > 0:
            fold = prev[-1] / prev[0]
            ax_cl.annotate(f'{fold:.0f}×',
                           xy=(ws_vals[-1], prev[-1]),
                           xytext=(8, 5), textcoords='offset points',
                           fontsize=12, fontweight='bold', color=ref_colors[ref])

    ax_cl.set_xlabel('Window size (±nt)', fontsize=18)
    ax_cl.set_ylabel('Prevalence (fraction of\npositions counted as positive)', fontsize=14)
    ax_cl.set_title('Prevalence inflation with window size',
                    fontsize=12, fontweight='bold', pad=8)
    ax_cl.set_xlim(-0.3, 10.3)
    ax_cl.set_xticks(range(11))
    ax_cl.tick_params(axis='both', labelsize=14)
    ax_cl.grid(alpha=0.2, lw=0.5)
    ax_cl.legend(fontsize=11, loc='upper left', framealpha=0.92,
                 edgecolor='#cccccc')

    # ── Panel c-right: Dumbbell — AUPRC(w=0) vs peak AUPRC ───────────────
    ax_cr = fig.add_subplot(gs[2, 1])

    for ref_idx, ref in enumerate(REFS):
        color = ref_colors[ref]
        y_off = -0.12 + ref_idx * 0.24
        for i, tool in enumerate(tools):
            if ref not in wdict.get(tool, {}) or 0 not in wdict[tool][ref]:
                continue
            d = wdict[tool][ref]
            auprc_w0 = d[0]['auprc']
            vals = {w: d[w]['auprc'] for w in WINDOWS if w in d}
            peak_w = max(vals, key=vals.get)
            auprc_peak = vals[peak_w]

            y = i + y_off
            # Connecting line
            ax_cr.plot([auprc_w0, auprc_peak], [y, y],
                       color=color, lw=1.5, alpha=0.6, zorder=2)
            # w=0 (open circle)
            ax_cr.scatter(auprc_w0, y, s=70, marker='o', facecolors='white',
                          edgecolors=color, linewidths=1.5, zorder=4)
            # Peak (filled circle)
            ax_cr.scatter(auprc_peak, y, s=70, marker='o', facecolors=color,
                          edgecolors='white', linewidths=0.8, zorder=4)
            # Label peak window if different from w=0
            if peak_w != 0:
                ax_cr.annotate(f'±{peak_w}',
                               xy=(auprc_peak, y), xytext=(5, 0),
                               textcoords='offset points', fontsize=8,
                               color=color, va='center', fontweight='bold')

    ax_cr.set_yticks(range(len(tools)))
    ax_cr.set_yticklabels(
        [TOOL_DISPLAY_NAMES.get(t, t) for t in tools], fontsize=14)
    ax_cr.set_xlabel('AUPRC', fontsize=18)
    ax_cr.set_title('Exact match (○) vs best window (●)',
                    fontsize=12, fontweight='bold', pad=8)
    ax_cr.set_xlim(-0.02, 1.02)
    ax_cr.tick_params(axis='x', labelsize=14)
    ax_cr.grid(axis='x', alpha=0.2, lw=0.5)
    ax_cr.invert_yaxis()

    legend_el = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='white',
               markeredgecolor='gray', markeredgewidth=1.5, markersize=9,
               label='Exact match (w=0)'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='gray',
               markeredgecolor='white', markersize=9,
               label='Best window'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#3C5488',
               markeredgecolor='white', markersize=8,
               label=REF_LABELS['16s_88_rrsE']),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#E64B35',
               markeredgecolor='white', markersize=8,
               label=REF_LABELS['23s_78_rrlB']),
    ]
    ax_cr.legend(handles=legend_el, fontsize=9, loc='lower right',
                 framealpha=0.92, edgecolor='#cccccc')

    fig.text(0.025, 0.31, 'c', fontsize=25, fontweight='bold', va='top')

    # ── Save ──────────────────────────────────────────────────────────────
    save_figure_pair(fig, save_prefix)
    plt.show()
    print(f"Saved: {save_prefix}.png and .svg")
    return fig

In [ ]:
# Generate Supplementary Figure S9 — Window Analysis at 1000x coverage

figS9 = plot_window_analysis(wdict, save_prefix='suppfigS9_window_analysis')

## All plots finished